# Experiment results — analysis, comparison and conclusions

Reads every experiment artifact this repository can produce, analyses what exists, and
**skips what does not** with an explicit note so nothing looks silently absent.

Everything is emitted twice: **inline in the cell output** and into **`comparison/`**

```
comparison/
├── REPORT.md              the full narrative report
├── CONCLUSIONS.md         findings ranked, with the claims that are safe to make
├── SKIPPED.md             what is missing, why, and what it would have answered
├── numbers.json           every number, machine-readable
├── paper_tables.tex       LaTeX, booktabs, ready to paste
├── availability.csv       experiment -> artifact -> available?
├── figures/*.png          every figure at 200 dpi
└── tables/*.md            every table as Markdown
```

**Nothing is retrained and no GPU is required.** This notebook only reads finished
artifacts, so a full *Run All* takes seconds.


---
## 0 · Setup

In [ ]:
import os, sys, json, glob, math, textwrap, warnings
from pathlib import Path
from datetime import datetime

# EDIT if this notebook is not sitting in the repository root.
REPO_PATH = "/home/ario/PY/3D-VG"
os.chdir(REPO_PATH)
REPO = Path.cwd()
if not (REPO / "scripts" / "ScanRefer_train.py").is_file():
    raise SystemExit(f"Run from the repository root. cwd={REPO}")

CHECKPOINT = "2024-12-18_20-40-38_3DVG-FIXED"
SPLIT      = "val"

# ---- output tree ---------------------------------------------------------------------
OUT     = REPO / "comparison"
FIGDIR  = OUT / "figures"
TABDIR  = OUT / "tables"
for d in (OUT, FIGDIR, TABDIR):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

# Figures must be legible pasted into a two-column paper, so keep the type large
# relative to the canvas and never rely on colour alone to carry a distinction.
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.grid": True, "grid.alpha": 0.25,
    "grid.linestyle": "-", "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "legend.frameon": False, "legend.fontsize": 9,
})

# Okabe-Ito: colour-blind safe, and distinguishable in greyscale print.
C = {"ours": "#0072B2", "base": "#D55E00", "gpt": "#0072B2", "llama": "#009E73",
     "spacy": "#D55E00", "none": "#999999", "warn": "#CC79A7", "ink": "#333333"}
HATCH = ["", "///", "...", "xxx"]

# ---- collectors ----------------------------------------------------------------------
NUMBERS  = {}     # machine-readable, -> comparison/numbers.json
REPORT   = []     # markdown blocks, -> comparison/REPORT.md
SKIPPED  = []     # (key, artifact, what it would answer, blocked_by)
FIGURES  = []     # (name, relpath, caption)
TABLES   = []     # (name, relpath, caption)
PVALUES  = []     # (label, p, family) -> multiple-comparison correction in section 9
CLAIMS   = []     # (strength, claim, evidence)

print(f"repo        {REPO}")
print(f"checkpoint  {CHECKPOINT}")
print(f"output      {OUT.relative_to(REPO)}/")
print(f"numpy {np.__version__} | matplotlib {matplotlib.__version__}")

In [ ]:
# ======================================================================================
# Helpers. Every emitter writes to comparison/ AND renders inline.
# ======================================================================================

def h(text, level=3):
    display(Markdown(f"{'#' * level} {text}"))

def note(text):
    display(Markdown(text))

def report(text):
    """Add a block to comparison/REPORT.md without printing it twice."""
    REPORT.append(text.strip("\n"))

def both(text, level=None):
    """Show inline and record in the report."""
    note(text); report(text)

def fig_save(fig, name, caption=""):
    path = FIGDIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    FIGURES.append((name, str(path.relative_to(REPO)), caption))
    plt.show()
    if caption:
        note(f"*Figure — {caption}*  \n`{path.relative_to(REPO)}`")

def mdtable(headers, rows, name=None, caption="", bold_first=True):
    """Render a Markdown table inline and save it. Rows are sequences of str."""
    def fmt(r):
        cells = [str(c) for c in r]
        if bold_first and cells:
            pass
        return "| " + " | ".join(cells) + " |"
    lines = ["| " + " | ".join(str(x) for x in headers) + " |",
             "|" + "|".join("---" for _ in headers) + "|"]
    lines += [fmt(r) for r in rows]
    text = "\n".join(lines)
    note(text)
    if caption:
        note(f"*{caption}*")
    if name:
        path = TABDIR / f"{name}.md"
        head = f"# {caption or name}\n\n" if (caption or name) else ""
        path.write_text(head + text + "\n")
        TABLES.append((name, str(path.relative_to(REPO)), caption))
        report(f"**{caption or name}**\n\n{text}")
    return text

def skip(key, artifact, answers, blocked_by):
    """Record a missing artifact and say so loudly, without failing."""
    SKIPPED.append(dict(key=key, artifact=artifact, answers=answers,
                        blocked_by=blocked_by))
    display(Markdown(
        f"> **SKIPPED — `{key}`**  \n"
        f"> missing: `{artifact}`  \n"
        f"> would answer: {answers}  \n"
        f"> blocked by: {blocked_by}  \n"
        f"> *This section will populate automatically once the artifact exists.*"))

def load_json(relpath):
    p = REPO / relpath
    if not p.is_file():
        return None
    try:
        return json.loads(p.read_text())
    except Exception as e:
        print(f"  [warn] {relpath} is not readable JSON: {e}")
        return None

def exists(pattern):
    """True if any path matches (glob allowed)."""
    return bool(glob.glob(str(REPO / pattern), recursive=True))

def pp(x, nd=2):
    """Fraction -> percentage points, or pass a percentage through."""
    return None if x is None else round(x * 100, nd) if abs(x) <= 1.5 else round(x, nd)

def stars(p):
    if p is None or (isinstance(p, float) and math.isnan(p)): return ""
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "n.s."

def wilson(k, n, z=1.96):
    """Wilson score interval -- correct near 0 and 1, unlike the normal approximation."""
    if not n: return (0.0, 0.0)
    p = k / n
    d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    hw = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return (max(0.0, c-hw), min(1.0, c+hw))

def cohen_h(p1, p2):
    """Effect size for two proportions (arcsine transform). 0.2 small / 0.5 med / 0.8 large."""
    if p1 is None or p2 is None: return None
    return abs(2*math.asin(math.sqrt(max(0, min(1, p1)))) -
               2*math.asin(math.sqrt(max(0, min(1, p2)))))

def holm(pairs, alpha=0.05):
    """Holm-Bonferroni. pairs = [(label, p)]. Returns [(label, p, p_adj, reject)]."""
    clean = [(l, p) for l, p in pairs if p is not None and not math.isnan(p)]
    m = len(clean)
    out, running = [], 0.0
    for i, (l, p) in enumerate(sorted(clean, key=lambda t: t[1])):
        adj = min(1.0, max(running, (m - i) * p))
        running = adj
        out.append((l, p, adj, adj < alpha))
    return out

def track_p(label, p, family="global"):
    if p is not None and not (isinstance(p, float) and math.isnan(p)):
        PVALUES.append((label, float(p), family))
    return p

def claim(strength, text, evidence):
    """strength: 'safe' | 'directional' | 'unsupported'"""
    CLAIMS.append((strength, text, evidence))

print("helpers ready")

---
## 1 · Availability — every experiment, checked

The registry below mirrors `run_one_experiment.ipynb` (41 experiments) and maps each to the
artifact it must produce. Everything downstream is gated on this table, so a missing result
produces a **SKIPPED** note rather than a crash or, worse, a silent gap.

In [ ]:
# key, phase, artifact glob, what it answers, reviewer points, blocker if absent
REGISTRY = [
 # ---- phase A: retraining (GPU) ----
 # `best.txt` is the completion marker. A run that crashes mid-training still leaves a
 # folder and an eval.txt behind, so globbing the folder alone would report a failed run
 # as a finished one -- which is exactly what happened to train-parser-gpt.
 ("train-no-copypaste","A","outputs/*ABL-NO-COPYPASTE*/best.txt","isolated contribution of proposal copy-paste","R2, R4.7","GPU training never run"),
 ("train-parser-gpt","A","outputs/*ABL-PARSER-GPT*/best.txt","variant A reference arm","R4.4","CRASHED -- ScanRefer_train.py rejected --num_workers 'auto'"),
 ("train-parser-spacy","A","outputs/*ABL-PARSER-SPACY*/best.txt","is an LLM parser needed at all?","R3.1, R2","GPU training never run"),
 ("train-parser-llama","A","outputs/*ABL-PARSER-LLAMA*/best.txt","open LLaMA-3 vs the GPT API","R2, R3.1","GPU training never run"),
 ("train-parser-none","A","outputs/*ABL-PARSER-NONE*/best.txt","parser removed, fusion untouched","R4.4","GPU training never run"),
 ("train-parser-smalllm","A","outputs/*ABL-PARSER-SMALLLM*/best.txt","0.5B local model in place of the API","R3.1, R4.5","GPU training never run"),
 ("train-seeds","A","outputs/*ABL-SEED*/best.txt","run-to-run spread, mean +/- std","R4.8","GPU training never run"),
 ("train-attention-sweep","A","outputs/ablation/attention_layer_sweep/sweep_summary.json","how many attention layers","R3.3","GPU training never run"),
 # ---- phase B: evaluation (GPU) ----
 ("evaluate-main","B",f"outputs/2024-12-18_20-40-38_3DVG-FIXED/predictions.p","predictions for every phase-C analysis","--","(present, from Dec 2024)"),
 ("corruption-sweep","B","outputs/2024-12-18_20-40-38_3DVG-FIXED/corruption/*/predictions.p","causal parse-error -> grounding","R2, R4.3","GPU eval never run"),
 ("eval-only-swap","B","outputs/2024-12-18_20-40-38_3DVG-FIXED/parser_swap/*/eval_stdout.txt","parser swapped at test time only","R4.4","GPU eval never run"),
 # ---- phase C: parse caches ----
 ("parse-spacy","C","data_parsing/spacy_parsing_tokenized/tokenized_parsed_result_val.json","variant B cache","R3.1","--"),
 ("parse-none","C","data_parsing/noparse_tokenized/tokenized_parsed_result_val.json","variant D cache","R4.4","--"),
 ("parse-corrupt","C","data_parsing/final_parsing_tokenized_corrupt_all_*/tokenized_parsed_result_val.json","corrupted caches for the propagation test","R4.3","--"),
 ("parse-smalllm","C","data_parsing/smalllm*_parsing_tokenized/tokenized_parsed_result_val.json","variant E cache","R3.1","never run (needs a GPU-ish pass)"),
 ("scene-cache","C","cached_scenes/meta.json","frozen-detector cache","--","--"),
 ("scene-cache-validate","C","outputs/reports/scene-cache-validate/result.json","cached == end-to-end to 1e-4","--","GPU never run"),
 # ---- phase C: parser accuracy ----
 ("target-acc-gpt4o-mini","C","outputs/parser_eval/target_accuracy_gpt4o-mini.json","target-field accuracy, GPT","R3.1","--"),
 ("target-acc-spacy","C","outputs/parser_eval/target_accuracy_spacy.json","target-field accuracy, spaCy","R3.1","--"),
 ("target-acc-llama","C","outputs/parser_eval/target_accuracy_llama.json","target-field accuracy, LLaMA","R3.1","--"),
 ("target-acc-none","C","outputs/parser_eval/target_accuracy_none.json","target-field accuracy, no parser","R4.4","--"),
 ("target-acc-smalllm","C","outputs/parser_eval/target_accuracy_smalllm.json","target-field accuracy, small LM","R3.1","no small-LM parse cache"),
 # ---- phase C: analyses ----
 ("results-table","C","outputs/analysis/results_table/results_table.json","the paper's main table","R1.4, R3.2, R4.9","--"),
 ("linguistic-complexity","C","outputs/analysis/linguistic_complexity/linguistic_complexity.json","accuracy vs linguistic complexity","R4.2","--"),
 ("parse-quality-split","C","outputs/analysis/parse_quality_split/parse_quality_split.json","grounding vs parse correctness","R4.3","--"),
 ("failure-cases","C","outputs/analysis/failure_cases/failure_cases.json","failure taxonomy","R2, R4","--"),
 ("failure-figures","C","experiments/analysis/figures/figures.json","the qualitative figure","R2, R4","--"),
 ("annotation-sheet","C","outputs/analysis/annotation/sampling_manifest.json","manual parse assessment sheets","R1.2, R4.3","--"),
 ("error-taxonomy","C","outputs/analysis/annotation/error_taxonomy.json","manual parse error rates","R1.2, R4.3","only gpt labelled, n=30"),
 ("error-propagation","C","outputs/analysis/parse_error_propagation/parse_error_propagation.md","do parse errors propagate causally","R2, R4.3","needs corruption-sweep"),
 ("seed-aggregate","C","outputs/reports/seed-aggregate/result.json","mean +/- std over seeds","R4.8","needs train-seeds"),
 ("parse-field-comparison","C","outputs/analysis/parse_field_comparison/parse_field_comparison.json","adjectives/neighbors across parsers","R1.2, R3.1","--"),
 ("complex-sentence-showdown","C","outputs/analysis/complex_sentence_showdown/complex_sentence_showdown.json","parsers on the hardest descriptions","R3.1, R4.2","--"),
 # ---- phase C: complexity ----
 ("parsing-offline-check","C","outputs/reports/parsing-offline-check/result.json","is parsing offline?","R4.5","--"),
 ("parsing-latency-spacy","C","outputs/reports/parsing-latency-spacy/result.json","per-query parse cost","R4.5","--"),
 ("complexity-model","C","outputs/complexity/complexity_report.json","FLOPs, peak memory, latency","R2, R4.5","--"),
 # ---- phase C: diagnostics ----
 ("verify-eq7","C","outputs/diagnostics/eq7_distance_bias.json","sign/normalisation of Eq. 7","R1.3, R4.6","--"),
 ("audit-scene-cache","C","outputs/diagnostics/scene_cache_audit_val.json","cache integrity + recall ceiling","--","--"),
 ("cached-eval-cpu","C","outputs/diagnostics/cached_eval_cpu.json","cached path without CUDA","--","--"),
 # ---- phase D: utilities ----
 ("predict","D","outputs/reports/predict/result.json","inference on new descriptions","--","never run"),
 ("visualize","D",f"outputs/2024-12-18_20-40-38_3DVG-FIXED/vis","rendered scenes for figures","R2","never run for this checkpoint"),
]

AVAIL = {}
rows = []
for key, phase, pat, answers, rev, blocker in REGISTRY:
    ok = exists(pat)
    AVAIL[key] = dict(ok=ok, pattern=pat, answers=answers, reviewer=rev, blocker=blocker,
                      phase=phase)
    rows.append((phase, key, "YES" if ok else "no", rev, pat if len(pat) < 58 else pat[:55] + "..."))

n_ok = sum(1 for v in AVAIL.values() if v["ok"])
n = len(AVAIL)

display(Markdown(f"### {n_ok} of {n} artifacts present  \n"
                 f"phase A {sum(1 for v in AVAIL.values() if v['phase']=='A' and v['ok'])}/8 · "
                 f"phase B {sum(1 for v in AVAIL.values() if v['phase']=='B' and v['ok'])}/3 · "
                 f"phase C {sum(1 for v in AVAIL.values() if v['phase']=='C' and v['ok'])}/28 · "
                 f"phase D {sum(1 for v in AVAIL.values() if v['phase']=='D' and v['ok'])}/2"))

mdtable(["phase", "experiment", "available", "reviewer", "artifact"], rows,
        name="availability",
        caption=f"Artifact availability, {n_ok}/{n} present "
                f"(generated {datetime.now():%Y-%m-%d %H:%M})")

# CSV for spreadsheets
import csv
with open(OUT / "availability.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["phase","key","available","reviewer","artifact","blocked_by","answers"])
    for key, phase, pat, answers, rev, blocker in REGISTRY:
        w.writerow([phase, key, int(AVAIL[key]["ok"]), rev, pat, blocker, answers])

NUMBERS["availability"] = {k: v["ok"] for k, v in AVAIL.items()}
NUMBERS["availability_summary"] = dict(present=n_ok, total=n)
print(f"\nwrote {(OUT/'availability.csv').relative_to(REPO)}")

In [ ]:
# Availability at a glance -- a figure for the progress narrative, not for the paper.
phases = ["A", "B", "C", "D"]
labels = {"A": "A · retraining\n(GPU)", "B": "B · evaluation\n(GPU)",
          "C": "C · post-processing\n(CPU)", "D": "D · utilities"}
have = [sum(1 for v in AVAIL.values() if v["phase"] == p and v["ok"]) for p in phases]
miss = [sum(1 for v in AVAIL.values() if v["phase"] == p and not v["ok"]) for p in phases]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6),
                               gridspec_kw={"width_ratios": [1, 1.5]})
y = np.arange(len(phases))
ax1.barh(y, have, color="#009E73", label="available", edgecolor="white")
ax1.barh(y, miss, left=have, color="#DDDDDD", label="missing", edgecolor="white")
ax1.set_yticks(y); ax1.set_yticklabels([labels[p] for p in phases], fontsize=8.5)
ax1.invert_yaxis(); ax1.set_xlabel("experiments")
ax1.set_title("Artifacts by phase")
for i, (a, b) in enumerate(zip(have, miss)):
    ax1.text(a + b + 0.4, i, f"{a}/{a+b}", va="center", fontsize=8.5)
ax1.legend(loc="lower right"); ax1.grid(axis="y", visible=False)

# Which reviewer points have supporting evidence on disk
rev_hit, rev_tot = {}, {}
for k, v in AVAIL.items():
    for r in [x.strip() for x in v["reviewer"].split(",") if x.strip() not in ("", "--")]:
        rev_tot[r] = rev_tot.get(r, 0) + 1
        rev_hit[r] = rev_hit.get(r, 0) + int(v["ok"])
order = sorted(rev_tot, key=lambda r: (-rev_tot[r], r))
x = np.arange(len(order))
frac = [rev_hit[r] / rev_tot[r] for r in order]
cols = ["#009E73" if f == 1 else "#E69F00" if f > 0 else "#D55E00" for f in frac]
ax2.bar(x, [rev_tot[r] for r in order], color="#EEEEEE", edgecolor="white")
ax2.bar(x, [rev_hit[r] for r in order], color=cols, edgecolor="white")
ax2.set_xticks(x); ax2.set_xticklabels(order, rotation=45, ha="right", fontsize=8.5)
ax2.set_ylabel("experiments"); ax2.set_title("Evidence on disk, per reviewer point")
ax2.grid(axis="x", visible=False)
ax2.text(0.99, 0.95, "green = complete · orange = partial · red = none",
         transform=ax2.transAxes, ha="right", va="top", fontsize=8, color=C["ink"])
fig.tight_layout()
fig_save(fig, "01_availability", "Artifact availability by phase, and evidence coverage per reviewer point")

---
## 2 · Main results (R1.4, R3.2, R4.9)

The headline comparison against 3DVG-Trans. Both columns are **recomputed on this machine
from `predictions.p`**, not quoted from the original papers — which is exactly what R4.9
asks for, and it must be stated in the table caption.

In [ ]:
rt = load_json("outputs/analysis/results_table/results_table.json")
if rt is None:
    skip("results-table", AVAIL["results-table"]["pattern"],
         "the paper's main table (R1.4, R3.2, R4.9)", AVAIL["results-table"]["blocker"])
else:
    res, subsets = rt["results"], ["overall", "unique", "multiple"]
    models = rt["models"]
    both(f"**split** `{rt['split']}` · **annotations** {rt['n']:,} "
         f"(unique {rt['subset_sizes']['unique']:,} / multiple {rt['subset_sizes']['multiple']:,})")

    rows = []
    for m in models:
        r = [f"**{m}**" if m == "ours" else m]
        for s in subsets:
            for t in ("acc_0.25", "acc_0.5"):
                v = res[m][s][t]; ci = res[m][s].get(f"{t}_ci")
                r.append(f"{pp(v):.2f} [{pp(ci[0]):.1f}–{pp(ci[1]):.1f}]" if ci else f"{pp(v):.2f}")
        rows.append(r)
    hdr = ["model"] + [f"{s} @{t}" for s in subsets for t in ("0.25", "0.5")]
    mdtable(hdr, rows, name="02_main_results",
            caption="Main results with 95% bootstrap CIs. All numbers recomputed locally "
                    "from predictions.p, not quoted from the source papers (R1.4, R3.2, R4.9).")

    # ---- paired significance: McNemar, already computed per subset ----
    ref = [m for m in models if m != "ours"]
    if ref:
        base = ref[0]
        vs = res[base].get("vs_ours", {})
        rows = []
        for s in subsets:
            for t in ("0.25", "0.5"):
                k = f"{s}_{t}"
                if k not in vs: continue
                d = vs[k]
                p = track_p(f"ours vs {base}, {s} @{t}", d["p_value"], "main")
                eff = cohen_h(res["ours"][s][f"acc_{t}"], res[base][s][f"acc_{t}"])
                rows.append([f"{s} @{t}", f"{d['diff_pp']:+.2f}", f"{d['mcnemar_chi2']:.1f}",
                             f"{d['p_value']:.2e}", stars(d["p_value"]),
                             f"{eff:.3f}" if eff else "--",
                             f"{d['reference_only']} / {d['other_only']}"])
        mdtable(["subset", "diff (pp)", "McNemar χ²", "p", "sig", "Cohen's h", "ours-only / base-only"],
                rows, name="02_main_significance",
                caption=f"Paired McNemar tests, ours vs {base}. The discordant-pair counts "
                        f"are the actual evidence: only annotations where the two models "
                        f"disagree carry information.")
        NUMBERS["main_vs_baseline"] = vs

    NUMBERS["main_results"] = {m: {s: res[m][s] for s in subsets} for m in models}

    # ---- where does the advantage live? ----
    if ref:
        u = res["ours"]["unique"]["acc_0.25"] - res[base]["unique"]["acc_0.25"]
        mu = res["ours"]["multiple"]["acc_0.25"] - res[base]["multiple"]["acc_0.25"]
        pu = vs.get("unique_0.25", {}).get("p_value")
        pm = vs.get("multiple_0.25", {}).get("p_value")
        both(f"""
**Reading.** At IoU 0.25 the gain over {base} is **{pp(mu):+.2f} pp on *multiple*** ({stars(pm)})
against **{pp(u):+.2f} pp on *unique*** ({stars(pu)}). The method's premise is adjacency
reasoning, which can only help when distractors exist — so the advantage appearing almost
entirely in the ambiguous subset is the pattern the method predicts, not an artifact.
State this explicitly; it is stronger evidence for the mechanism than the overall number.
""")
        claim("safe",
              f"The improvement over {base} is concentrated in the *multiple* subset "
              f"({pp(mu):+.2f} pp, p={pm:.1e}) rather than *unique* ({pp(u):+.2f} pp, n.s.), "
              f"consistent with the adjacency-reasoning mechanism.",
              "outputs/analysis/results_table/results_table.json")

In [ ]:
if rt is not None:
    res, models = rt["results"], rt["models"]
    subsets = ["overall", "unique", "multiple"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.9), sharey=False)
    for ax, thr in zip(axes, ["0.25", "0.5"]):
        x = np.arange(len(subsets)); w = 0.36
        for i, m in enumerate(models):
            vals = [pp(res[m][s][f"acc_{thr}"]) for s in subsets]
            los, his = [], []
            for s in subsets:
                ci = res[m][s].get(f"acc_{thr}_ci")
                v = pp(res[m][s][f"acc_{thr}"])
                los.append(v - pp(ci[0]) if ci else 0); his.append(pp(ci[1]) - v if ci else 0)
            ax.bar(x + (i - 0.5) * w, vals, w,
                   color=C["ours"] if m == "ours" else C["base"],
                   hatch=HATCH[i], edgecolor="white",
                   label=m, yerr=[los, his], capsize=3,
                   error_kw=dict(lw=1, ecolor=C["ink"]))
            for xi, v in zip(x + (i - 0.5) * w, vals):
                ax.text(xi, v + max(his) + 0.8, f"{v:.1f}", ha="center", fontsize=8)
        ax.set_xticks(x); ax.set_xticklabels(subsets)
        ax.set_ylabel("Acc@%s (%%)" % thr); ax.set_title(f"IoU ≥ {thr}")
        ax.set_ylim(0, max(pp(res[m][s][f'acc_{thr}']) for m in models for s in subsets) * 1.25)
        ax.grid(axis="x", visible=False)
    axes[0].legend(loc="upper left")
    fig.suptitle("Grounding accuracy by subset, with 95% bootstrap CIs", y=1.02, fontsize=11.5, fontweight="bold")
    fig.tight_layout()
    fig_save(fig, "02_main_results", "Main results. Error bars are 95% bootstrap CIs; the "
                                     "advantage is concentrated in the multiple subset.")

---
## 2b · The ablation arms — retrained models (R4.4, R4.7, R4.8)

**This is the section the revision turns on.** Phase-A training has now run, so the
questions that no amount of post-hoc analysis could answer are answerable.

Three things must be established before any number here is read:

1. **These runs use a different protocol from the headline table in §2.** They train on the
   **frozen-detector cache** for **20 epochs from a warm start**; the December 2024
   checkpoint was end-to-end for 100 epochs. Ablation arms are comparable **to each other**
   — which is all an ablation needs — but must **never** be quoted beside the 48.91 figure.
2. **The reference arm crashed.** `ABL-PARSER-GPT` has no `best.txt`. The two seed runs use
   the identical configuration (GPT parses, `final_parsing_tokenized`), so **the seed runs
   serve as the reference**, and they also give the noise floor.
3. **Every delta is judged against seed noise**, not against zero. A gap smaller than the
   run-to-run spread is not a finding — that is precisely R4.8's complaint.

In [ ]:
import re

def parse_best(path):
    """Pull the scores out of a run's best.txt. Returns None if the run never finished."""
    p = REPO / path
    if not p.is_file():
        return None
    t = p.read_text()
    out = {}
    m = re.search(r"\[best\] epoch:\s*(\d+)", t)
    if m: out["best_epoch"] = int(m.group(1))
    m = re.search(r"iou_rate_0\.25:\s*([0-9.]+),\s*iou_rate_0\.5:\s*([0-9.]+)", t)
    if m: out["acc_0.25"], out["acc_0.5"] = float(m.group(1)), float(m.group(2))
    for k in ("ref_acc", "obj_acc", "lang_acc"):
        m = re.search(rf"\[sco\.\]\s*{k}:\s*([0-9.]+)", t)
        if m: out[k] = float(m.group(1))
    return out or None

# arm key -> (folder glob, parser cache, what it isolates)
ARMS = [
    ("ABL-SEED1",        "outputs/*ABL-SEED1*",        "gpt4o-mini", "reference, seed 1"),
    ("ABL-SEED2",        "outputs/*ABL-SEED2*",        "gpt4o-mini", "reference, seed 2"),
    ("ABL-PARSER-GPT",   "outputs/*ABL-PARSER-GPT*",   "gpt4o-mini", "reference, seed 42"),
    ("ABL-PARSER-SPACY", "outputs/*ABL-PARSER-SPACY*", "spacy",      "rule-based parser (R3.1)"),
    ("ABL-PARSER-NONE",  "outputs/*ABL-PARSER-NONE*",  "none",       "no parser at all (R4.4)"),
    ("ABL-PARSER-LLAMA", "outputs/*ABL-PARSER-LLAMA*", "llama",      "open LLM (R2, R3.1)"),
    ("ABL-PARSER-SMALLLM","outputs/*ABL-PARSER-SMALLLM*","smalllm",  "0.5B local model (R3.1)"),
    ("ABL-NO-COPYPASTE", "outputs/*ABL-NO-COPYPASTE*", "gpt4o-mini", "copy-paste OFF (R2, R4.7)"),
]

RUNS, INCOMPLETE = {}, []
for key, pat, parser, what in ARMS:
    hits = sorted(glob.glob(str(REPO / pat)))
    if not hits:
        continue
    folder = Path(hits[-1])
    b = parse_best(folder.relative_to(REPO) / "best.txt")
    if b is None:
        INCOMPLETE.append((key, folder.name, what))
    else:
        b.update(folder=folder.name, parser=parser, what=what)
        RUNS[key] = b

print(f"{len(RUNS)} completed arm(s), {len(INCOMPLETE)} incomplete\n")
for k, v in RUNS.items():
    print(f"  {k:22s} @0.25={pp(v['acc_0.25']):.3f}  @0.5={pp(v['acc_0.5']):.3f}  "
          f"best_epoch={v.get('best_epoch')}")
for k, f, w in INCOMPLETE:
    print(f"  {k:22s} INCOMPLETE (no best.txt) -- {f}")

In [ ]:
if not RUNS:
    skip("phase-A ablations", "outputs/*ABL*/best.txt",
         "the controlled parser ablation, copy-paste isolation and seed spread "
         "(R4.4, R4.7, R4.8)", "no training arm has finished")
else:
    # ---- the noise floor -------------------------------------------------------------
    seeds = {k: v for k, v in RUNS.items() if k.startswith("ABL-SEED")}
    ref_mean = ref_sd = None
    if len(seeds) >= 2:
        vals25 = [v["acc_0.25"] for v in seeds.values()]
        vals50 = [v["acc_0.5"] for v in seeds.values()]
        ref_mean, ref_sd = float(np.mean(vals25)), float(np.std(vals25, ddof=1))
        m50, s50 = float(np.mean(vals50)), float(np.std(vals50, ddof=1))
        rng = max(vals25) - min(vals25)
        both(f"""
### The noise floor first (R4.8)

`{'`, `'.join(seeds)}` are the **same configuration under different seeds**:

| | Acc@0.25 | Acc@0.5 |
|---|---|---|
| mean ± sd (n={len(seeds)}) | **{pp(ref_mean):.2f} ± {pp(ref_sd):.2f}** | {pp(m50):.2f} ± {pp(s50):.2f} |
| range | {pp(rng):.2f} pp | {pp(max(vals50)-min(vals50)):.2f} pp |

**Run-to-run spread is ±{pp(ref_sd):.2f} pp.** This is the number R4.8 asked for, and it is
now the yardstick: any ablation gap below roughly **{pp(2*ref_sd):.2f} pp** (2 sd) is
indistinguishable from noise.

> **n = 2.** A standard deviation from two runs is a very weak estimate — the
> `aggregate_seed_results.py` output says so itself. A third seed would materially
> strengthen every claim below, and it is the cheapest remaining experiment.
""")
        NUMBERS["seed_spread"] = dict(n=len(seeds), mean_25=ref_mean, sd_25=ref_sd,
                                      mean_5=m50, sd_5=s50, range_25=rng)
        claim("safe", f"Run-to-run spread of the main configuration is "
                      f"{pp(ref_mean):.2f} ± {pp(ref_sd):.2f} pp Acc@0.25 over {len(seeds)} seeds.",
              "outputs/*ABL-SEED*/best.txt")

    # ---- the arms table ---------------------------------------------------------------
    order = [k for k, *_ in ARMS if k in RUNS]
    rows = []
    for k in order:
        v = RUNS[k]
        d25 = (v["acc_0.25"] - ref_mean) if ref_mean is not None else None
        d50 = None
        if ref_mean is not None:
            verdict = ("reference" if k.startswith("ABL-SEED") else
                       "**within noise**" if abs(pp(d25)) < 2 * pp(ref_sd) else
                       ("**better**" if d25 > 0 else "**worse**"))
            nsd = abs(d25) / ref_sd if ref_sd else float("inf")
        else:
            verdict, nsd = "--", None
        rows.append([f"**{k}**", v["parser"], v["what"],
                     f"{pp(v['acc_0.25']):.2f}", f"{pp(v['acc_0.5']):.2f}",
                     f"{pp(d25):+.2f}" if d25 is not None else "--",
                     f"{nsd:.1f}σ" if nsd is not None and np.isfinite(nsd) else "--",
                     verdict, v.get("best_epoch")])
    mdtable(["arm", "parser", "isolates", "Acc@0.25", "Acc@0.5", "Δ vs ref (pp)",
             "|Δ| in sd", "verdict", "best epoch"], rows, name="02b_ablation_arms",
            caption="Phase-A ablation arms, frozen-detector protocol, 20 epochs from a warm "
                    "start. Δ is against the seed-run mean. Not comparable to §2.")
    NUMBERS["ablation_arms"] = {k: RUNS[k] for k in order}

    if INCOMPLETE:
        for k, f, w in INCOMPLETE:
            skip(k, f"outputs/{f}/best.txt", f"{w}",
                 "the run crashed before writing best.txt")


In [ ]:
if RUNS and ref_mean is not None:
    order = [k for k, *_ in ARMS if k in RUNS]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.1),
                                 gridspec_kw={"width_ratios": [1.25, 1]})

    # --- absolute accuracy, with the seed band drawn behind ---
    xs = np.arange(len(order))
    vals = [pp(RUNS[k]["acc_0.25"]) for k in order]
    cols = [C["ours"] if k.startswith("ABL-SEED") else
            C["warn"] if "COPYPASTE" in k else
            C["spacy"] if "SPACY" in k else
            C["none"] if "NONE" in k else C["llama"] for k in order]
    a1.axhspan(pp(ref_mean - 2*ref_sd), pp(ref_mean + 2*ref_sd), color="#0072B2", alpha=0.13,
               zorder=0, label="reference ±2 sd (seed noise)")
    a1.axhline(pp(ref_mean), color=C["ours"], ls="--", lw=1.3, zorder=1)
    a1.bar(xs, vals, 0.62, color=cols, edgecolor="white", zorder=2)
    for x, k, v in zip(xs, order, vals):
        a1.text(x, v + 0.06, f"{v:.2f}", ha="center", fontsize=8.5, zorder=3)
    a1.set_xticks(xs)
    a1.set_xticklabels([k.replace("ABL-", "").replace("PARSER-", "") for k in order],
                       rotation=20, ha="right", fontsize=8.5)
    a1.set_ylabel("Acc@0.25 (%)")
    a1.set_ylim(min(vals) - 0.8, max(vals) + 0.5)
    a1.set_title("Ablation arms vs the seed-noise band")
    a1.legend(loc="lower left", fontsize=8); a1.grid(axis="x", visible=False)

    # --- deltas, in units of seed sd ---
    comp = [k for k in order if not k.startswith("ABL-SEED")]
    if comp:
        d = [pp(RUNS[k]["acc_0.25"] - ref_mean) for k in comp]
        nsd = [abs(x) / pp(ref_sd) for x in d]
        ys = np.arange(len(comp))
        bc = ["#009E73" if x > 0 else C["base"] for x in d]
        a2.barh(ys, d, 0.55, color=bc, edgecolor="white")
        a2.axvline(0, color=C["ink"], lw=1)
        for s, lbl in [(2, "2 sd"), (-2, None)]:
            a2.axvline(s * pp(ref_sd), color=C["ours"], ls=":", lw=1.2,
                       label=("±2 sd of seed noise" if lbl else None))
        for y, (k, x, n) in enumerate(zip(comp, d, nsd)):
            a2.text(x + (0.03 if x > 0 else -0.03), y, f"{x:+.2f} ({n:.0f}σ)",
                    va="center", ha="left" if x > 0 else "right", fontsize=8.5)
        a2.set_yticks(ys)
        a2.set_yticklabels([k.replace("ABL-", "").replace("PARSER-", "") for k in comp],
                           fontsize=8.5)
        a2.invert_yaxis()
        a2.set_xlabel("Δ Acc@0.25 vs reference (pp)")
        a2.set_title("Is the gap bigger than the noise?")
        a2.set_xlim(min(d) * 1.7 if min(d) < 0 else -0.3, max(max(d) * 1.7, 0.3))
        a2.legend(loc="lower right", fontsize=8); a2.grid(axis="y", visible=False)
    fig.tight_layout()
    fig_save(fig, "02b_ablation_arms",
             "Retrained ablation arms against the seed-noise band. Bars outside the shaded "
             "band exceed run-to-run variation (R4.4, R4.7, R4.8).")

In [ ]:
if RUNS and ref_mean is not None:
    # ---- R4.4: the parser ablation the paper's Table 6 could not deliver ----
    parser_arms = {RUNS[k]["parser"]: RUNS[k] for k in RUNS if "PARSER" in k}
    if "spacy" in parser_arms or "none" in parser_arms:
        h("R4.4 — the controlled parser comparison", 4)
        lines = []
        for lab in ("spacy", "none", "llama", "smalllm"):
            if lab not in parser_arms: continue
            v = parser_arms[lab]
            d = pp(v["acc_0.25"] - ref_mean); n = abs(d) / pp(ref_sd)
            lines.append(f"- **{lab}**: {pp(v['acc_0.25']):.2f}%, "
                         f"**{d:+.2f} pp** vs the GPT reference ({n:.0f}× the seed sd)")
        both(f"""
Fusion architecture held fixed, **only the parse cache changed** — exactly the design R4.4
asked for and the published Table 6 did not provide:

{chr(10).join(lines)}

**Reading.** The ordering matches the method's story and every gap clears the noise floor:
replacing GPT-4o-mini with a rule-based parser costs
{abs(pp(parser_arms['spacy']['acc_0.25'] - ref_mean)):.2f} pp, and removing the parser
entirely costs {abs(pp(parser_arms['none']['acc_0.25'] - ref_mean)):.2f} pp. The parser
contributes a small, consistent, measurable gain.
""" if "spacy" in parser_arms and "none" in parser_arms else "\n".join(lines))
        if "spacy" in parser_arms and "none" in parser_arms:
            claim("safe",
                  f"With the fusion architecture fixed, swapping GPT-4o-mini for spaCy costs "
                  f"{abs(pp(parser_arms['spacy']['acc_0.25']-ref_mean)):.2f} pp and removing the "
                  f"parser entirely costs {abs(pp(parser_arms['none']['acc_0.25']-ref_mean)):.2f} pp "
                  f"Acc@0.25, both exceeding the ±{pp(ref_sd):.2f} pp seed noise.",
                  "outputs/*ABL-PARSER-*/best.txt")

    # ---- R4.7: copy-paste ----
    if "ABL-NO-COPYPASTE" in RUNS:
        h("R4.7 — isolating the copy-paste augmentation", 4)
        v = RUNS["ABL-NO-COPYPASTE"]
        d = pp(v["acc_0.25"] - ref_mean); n = abs(d) / pp(ref_sd)
        direction = "HIGHER" if d > 0 else "lower"
        both(f"""
Copy-paste **disabled**, everything else held fixed: **{pp(v['acc_0.25']):.2f}%**, i.e.
**{d:+.2f} pp** against the reference ({n:.0f}× the seed sd).

R2 and R4.7 asked for the augmentation's isolated contribution. It measures as
**{d:+.2f} pp** — the augmentation did not help — and the gap clears the noise floor.
""")
        claim("safe",
              f"Disabling proposal copy-paste changed Acc@0.25 by {d:+.2f} pp "
              f"({n:.0f}x the seed sd): its isolated contribution is not positive.",
              "outputs/*ABL-NO-COPYPASTE*/best.txt")


---
## 3 · Linguistic complexity (R4.2)

R4.2 objects that *"results are not analyzed by sentence length or the number and type of
spatial relations. The existing Unique/Multiple split reflects object ambiguity rather than
linguistic complexity."* This is the direct answer — and it is also **the single most
over-claimable result in the set**, so the widening test is reported explicitly.

In [ ]:
lc = load_json("outputs/analysis/linguistic_complexity/linguistic_complexity.json")
if lc is None:
    skip("linguistic-complexity", AVAIL["linguistic-complexity"]["pattern"],
         "accuracy vs linguistic complexity (R4.2)", AVAIL["linguistic-complexity"]["blocker"])
else:
    both(f"**split** `{lc['split']}` · **annotations** {lc['num_annotations']:,} · "
         f"**IoU threshold** {lc['threshold']}")
    metrics = [k for k in lc["results"] if k != "relation_types"]
    NUMBERS["linguistic_complexity"] = {}

    for mk in metrics:
        blk = lc["results"][mk]
        h(f"{mk} — {blk['label']}", 4)
        models = list(blk["bins"][0]["models"])
        rows = []
        for b in blk["bins"]:
            r = [b["bin"], f"{b['n']:,}"]
            for m in models:
                v = b["models"][m]["acc_0.25"]; ci = b["models"][m].get("acc_0.25_ci")
                r.append(f"{pp(v):.2f} [{pp(ci[0]):.1f}–{pp(ci[1]):.1f}]" if ci else f"{pp(v):.2f}")
            if len(models) > 1:
                r.append(f"{pp(b['models'][models[0]]['acc_0.25'] - b['models'][models[1]]['acc_0.25']):+.2f}")
            rows.append(r)
        hdr = ["bin", "n"] + [f"{m} Acc@0.25" for m in models] + \
              ([f"gap {models[0]}−{models[1]}"] if len(models) > 1 else [])
        mdtable(hdr, rows, name=f"03_complexity_{mk}",
                caption=f"Accuracy by {blk['label']} (R4.2)")

        f = blk.get("findings", {})
        tr = f.get("trends", {}).get("ours", {})
        if tr:
            p = track_p(f"trend: accuracy vs {mk}", tr.get("p_value"), "complexity")
            both(f"- **Trend (ours)**: Spearman ρ = {tr['spearman_rho']:+.4f}, "
                 f"p = {tr['p_value']:.2e} ({stars(p)}), n = {tr['n']:,} — accuracy "
                 f"{'falls' if tr['spearman_rho'] < 0 else 'rises'} as {blk['label']} grows.")
        for base, g in f.get("gaps", {}).items():
            wd = g.get("widening", {})
            det = g.get("per_bin_detail", [])
            if det:
                seq = "  ".join(f"{pp(d['gap']):+.2f}{'*' if d['p_value'] < 0.05 else ''}" for d in det)
                nsig = sum(1 for d in det if d["p_value"] < 0.05)
                both(f"- **Per-bin gaps vs {base}** (McNemar): {seq}  → {nsig}/{len(det)} significant")
                for i, d in enumerate(det):
                    track_p(f"{mk} bin{i+1} gap vs {base}", d["p_value"], "complexity")
            if wd.get("available") and wd.get("difference") is not None:
                ci = wd.get("ci") or [None, None]
                lo, hi = ci[0], ci[1]
                sig = wd.get("significant")
                d_pp = pp(wd["difference"])
                ci_txt = (f"95% CI [{pp(lo):+.2f}, {pp(hi):+.2f}]"
                          if lo is not None else "CI unavailable")
                both(f"- **Widening test** ({wd.get('high_bin')} minus {wd.get('low_bin')}, "
                     f"{wd.get('resamples','?')} bootstrap resamples): "
                     f"**{d_pp:+.2f} pp**, {ci_txt} — "
                     + ("**the CI excludes zero** — the widening is significant."
                        if sig else
                        "**the CI includes zero.** The trend is directional only. Write "
                        "*'consistent with'*, never *'demonstrates'*, and never quote the "
                        "point estimate alone."))
                claim("safe" if sig else "directional",
                      f"The advantage over {base} widens with {blk['label']} "
                      f"({d_pp:+.2f} pp, {ci_txt}).",
                      "outputs/analysis/linguistic_complexity/linguistic_complexity.json")
        NUMBERS["linguistic_complexity"][mk] = blk

In [ ]:
if lc is not None:
    metrics = [k for k in lc["results"] if k != "relation_types"]
    ncol = min(2, len(metrics)); nrow = math.ceil(len(metrics) / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 3.5 * nrow), squeeze=False)
    for ax, mk in zip(axes.ravel(), metrics):
        blk = lc["results"][mk]
        models = list(blk["bins"][0]["models"])
        xs = np.arange(len(blk["bins"]))
        for i, m in enumerate(models):
            v = [pp(b["models"][m]["acc_0.25"]) for b in blk["bins"]]
            lo = [pp(b["models"][m]["acc_0.25"]) - pp(b["models"][m]["acc_0.25_ci"][0]) for b in blk["bins"]]
            hi = [pp(b["models"][m]["acc_0.25_ci"][1]) - pp(b["models"][m]["acc_0.25"]) for b in blk["bins"]]
            ax.errorbar(xs, v, yerr=[lo, hi], marker="os"[i % 2], ms=6, lw=2, capsize=3,
                        color=C["ours"] if m == "ours" else C["base"], label=m)
        # significance marks on the gap
        det = blk.get("findings", {}).get("gaps", {})
        if det:
            d0 = list(det.values())[0].get("per_bin_detail", [])
            for xi, d in zip(xs, d0):
                if d["p_value"] < 0.05:
                    top = max(pp(b["models"][m]["acc_0.25_ci"][1]) for m in models
                              for b in [blk["bins"][xi]])
                    ax.text(xi, top + 1.2, "*", ha="center", fontsize=13, color=C["ink"])
        ax.set_xticks(xs); ax.set_xticklabels([b["bin"] for b in blk["bins"]])
        ax.set_title(f"{blk['label']}"); ax.set_ylabel("Acc@0.25 (%)")
        ax.set_xlabel(f"n = {', '.join(f'{b[chr(110)]:,}' for b in blk['bins'])}", fontsize=8)
        ax.grid(axis="x", visible=False)
    for ax in axes.ravel()[len(metrics):]: ax.axis("off")
    axes[0][0].legend(loc="lower left")
    fig.suptitle("Accuracy by linguistic complexity  ( * = per-bin gap significant, McNemar p<0.05 )",
                 y=1.01, fontsize=11.5, fontweight="bold")
    fig.tight_layout()
    fig_save(fig, "03_linguistic_complexity",
             "Accuracy vs linguistic complexity (R4.2). Both models degrade; per-bin gaps "
             "are mostly significant but the widening trend itself is not.")

    rel = lc["results"].get("relation_types", {}).get("counts", {})
    if rel:
        top = sorted(rel.items(), key=lambda kv: -kv[1])[:12]
        fig, ax = plt.subplots(figsize=(7.4, 3.4))
        ax.bar([k for k, _ in top], [v for _, v in top], color=C["ours"], edgecolor="white")
        ax.set_ylabel("annotations"); ax.set_title("Spatial relation types in the split (R4.2)")
        ax.tick_params(axis="x", rotation=45)
        for lb in ax.get_xticklabels(): lb.set_ha("right")
        ax.grid(axis="x", visible=False)
        fig.tight_layout()
        fig_save(fig, "03_relation_types", "Distribution of spatial relation types (R4.2 asks "
                                           "for 'the number and type of spatial relations').")
        NUMBERS["relation_types"] = rel

---
## 4 · Is the LLM parser necessary? (R2, R3.1, R4.4)

The most consequential question in the reviews. R3.1: *"unclear whether similar performance
could be achieved using conventional dependency parsers or lightweight language models."*
R2 asks for the same ablation.

**What can and cannot be concluded here.** The grounding-level answer needs the five
retrained arms, which do not exist. What *does* exist is a complete **parse-level**
comparison — target accuracy against free ground truth, plus coverage and agreement on the
two ungrounded fields. That is real evidence, and it is worth reporting as such, but it is
not a substitute for the end-task numbers.

In [ ]:
PARSER_LABELS = ["gpt4o-mini", "llama", "spacy", "none", "smalllm"]
ta, missing_ta = {}, []
for lab in PARSER_LABELS:
    d = load_json(f"outputs/parser_eval/target_accuracy_{lab}.json")
    if d is None: missing_ta.append(lab)
    else: ta[lab] = d

if missing_ta:
    for lab in missing_ta:
        k = f"target-acc-{lab}"
        if k in AVAIL:
            skip(k, AVAIL[k]["pattern"], f"target-field accuracy for `{lab}` (R3.1)",
                 AVAIL[k]["blocker"])

if ta:
    h("4.1 · Target-field accuracy — free ground truth (R3.1)", 4)
    both("ScanRefer's `object_name` grounds the `target` field, so this is the one parser "
         "metric with real ground truth over the whole split. `fuzzy` is the honest column: "
         "most errors are synonyms (`trash can`→`bin`, `couch`→`sofa`).")
    rows = []
    for lab, d in sorted(ta.items(), key=lambda kv: -kv[1]["splits"]["train"]["fuzzy_pct"]):
        s = d["splits"]["train"]
        lo, hi = wilson(s["fuzzy"], s["total"])
        rows.append([f"**{lab}**", f"{s['total']:,}", f"{s['exact_pct']:.2f}",
                     f"{s['substring_pct']:.2f}", f"**{s['fuzzy_pct']:.2f}**",
                     f"[{pp(lo):.2f}–{pp(hi):.2f}]", s["no_target"]])
    mdtable(["parser", "n", "exact %", "substring %", "fuzzy %", "95% CI (fuzzy)", "no target"],
            rows, name="04_target_accuracy",
            caption="Target-field extraction accuracy on train, with Wilson CIs (R3.1).")
    NUMBERS["target_accuracy"] = {k: v["splits"] for k, v in ta.items()}

    real = {k: v for k, v in ta.items() if k != "none"}
    if len(real) >= 2:
        best = max(real, key=lambda k: real[k]["splits"]["train"]["fuzzy_pct"])
        rank = sorted(real, key=lambda k: -real[k]["splits"]["train"]["fuzzy_pct"])
        gap_sp = (real[rank[0]]["splits"]["train"]["fuzzy_pct"]
                  - real["spacy"]["splits"]["train"]["fuzzy_pct"]) if "spacy" in real else None
        both(f"""
**Reading.** `{best}` leads on fuzzy target accuracy
({real[best]['splits']['train']['fuzzy_pct']:.2f}%).""" + (f"""
Two findings worth stating plainly in the response letter:

1. **LLaMA-3 is not worse than the GPT API** ({real.get('llama',{}).get('splits',{}).get('train',{}).get('fuzzy_pct',float('nan')):.2f}% vs
   {real.get('gpt4o-mini',{}).get('splits',{}).get('train',{}).get('fuzzy_pct',float('nan')):.2f}%). An open model can replace the paid API on
   this field, which addresses R2/R3.1's cost-and-dependency concern directly.
2. **spaCy trails by {gap_sp:.2f} pp**, and its errors differ in *kind*: it emits the generic
   head noun (`chair`→`object`, ×132) rather than a synonym. That is a qualitative
   difference, not a tuning gap.
""" if gap_sp is not None and "llama" in real and "gpt4o-mini" in real else ""))
        if "llama" in real and "gpt4o-mini" in real:
            claim("safe",
                  f"On target extraction LLaMA-3 ({real['llama']['splits']['train']['fuzzy_pct']:.2f}%) "
                  f"matches or exceeds GPT-4o-mini ({real['gpt4o-mini']['splits']['train']['fuzzy_pct']:.2f}%), "
                  "so the paid API is not required for this field.",
                  "outputs/parser_eval/target_accuracy_*.json")
        if gap_sp is not None:
            claim("safe",
                  f"spaCy trails the LLM parsers by {gap_sp:.2f} pp on target extraction and "
                  "fails differently (generic head noun rather than synonym).",
                  "outputs/parser_eval/target_accuracy_*.json")

In [ ]:
pf = load_json("outputs/analysis/parse_field_comparison/parse_field_comparison.json")
if pf is None:
    skip("parse-field-comparison", AVAIL["parse-field-comparison"]["pattern"],
         "adjectives/neighbors coverage across parsers (R1.2, R3.1)",
         AVAIL["parse-field-comparison"]["blocker"])
else:
    h("4.2 · The ungrounded fields — coverage, faithfulness, agreement", 4)
    both(f"**split** `{pf['split']}` · **annotations** {pf['annotations']:,} · "
         f"**parsers** {', '.join(pf['parsers'])}")
    both("`adjectives` and `neighbors` have **no ground truth** — ScanRefer only grounds "
         "`target`. Every metric below is therefore reference-free: how often a parser "
         "declines the slot, how much of what it emits is copied from the source, and how "
         "much the parsers agree with each other.")
    if len(pf["parsers"]) < 3:
        note(f"> **Note.** Only {len(pf['parsers'])} parsers are covered "
             f"({', '.join(pf['parsers'])}). A cache for the missing parser exists, so "
             f"re-running `parse-field-comparison` with all of them is cheap (CPU, seconds) "
             f"and would strengthen this table.")
    NUMBERS["parse_fields"] = pf["fields"]
    for field, blk in pf["fields"].items():
        rows = []
        for name, s in blk["parsers"].items():
            ci = s.get("empty_ci", [None, None])
            rows.append([f"**{name}**", f"{pp(s['empty_rate']):.1f}",
                         f"[{pp(ci[0]):.1f}–{pp(ci[1]):.1f}]" if ci[0] is not None else "--",
                         f"{s['mean_tokens_when_present']:.2f}", f"{s['mean_phrases']:.2f}",
                         f"{pp(s['faithfulness']):.1f}",
                         f"{blk['consensus'].get(name, float('nan')):.3f}"])
        mdtable(["parser", "declined %", "95% CI", "tokens when present", "phrases",
                 "faithful %", "consensus"], rows, name=f"04_field_{field}",
                caption=f"`{field}` — reference-free parser comparison (R1.2, R3.1)")
        pw = ", ".join(f"{k} = {v:.3f}" for k, v in blk["pairwise"].items())
        both(f"- `{field}` pairwise Jaccard agreement: {pw}")
    both("""
> **Do not quote spaCy's faithfulness on its own.** spaCy is a copy-only parser: it cannot
> invent a token, so near-100% faithfulness is *near-tautological* for it and says nothing
> about quality. It must always be read beside the coverage column, where it is roughly
> twice as bad.
""")
    claim("unsupported",
          "spaCy's high faithfulness indicates good parse quality.",
          "It is a copy-only parser, so faithfulness is near-tautological; read coverage instead.")

In [ ]:
cs = load_json("outputs/analysis/complex_sentence_showdown/complex_sentence_showdown.json")
if cs is None:
    skip("complex-sentence-showdown", AVAIL["complex-sentence-showdown"]["pattern"],
         "parsers on the hardest descriptions (R3.1, R4.2)",
         AVAIL["complex-sentence-showdown"]["blocker"])
else:
    h("4.3 · Do parsers degrade on hard sentences? (R3.1 × R4.2)", 4)
    both(f"**ranked by** `{cs['rank_by']}` · **annotations** {cs['annotations']:,} · "
         f"**{len(cs['cases'])} hardest cases** inspected individually")
    NUMBERS["showdown_quartiles"] = cs["quartiles"]
    for field, per in cs["quartiles"].items():
        rows = []
        for name, qs in per.items():
            r = [f"**{name}**"] + [f"{pp(q['empty_rate']):.1f}" for q in qs]
            r.append(f"{pp(qs[-1]['empty_rate']) - pp(qs[0]['empty_rate']):+.1f}")
            rows.append(r)
        hdr = ["parser"] + [q["quartile"] for q in list(per.values())[0]] + ["Q4−Q1"]
        mdtable(hdr, rows, name=f"04_quartiles_{field}",
                caption=f"`{field}` declined-rate by complexity quartile (% empty)")
    both("""
**The honest reading, and it is not the intuitive one.** spaCy does **not** degrade faster
with complexity — its Q4−Q1 slope is *flatter* than the LLM parsers'. The damning number is
the **level, not the slope**: spaCy starts roughly twice as bad and stays there.

Claiming *"spaCy collapses on complex sentences"* would be **unsupported by this data** and
a reviewer checking the quartiles would catch it. The defensible claim is that spaCy is
**uniformly** worse across the whole complexity range.
""")
    claim("unsupported", "spaCy degrades faster than LLM parsers as sentences get harder.",
          "Its Q4−Q1 slope is flatter; the gap is a constant level difference.")
    claim("safe", "spaCy leaves the attribute slot empty about twice as often as the LLM "
                  "parsers at every complexity level, not just on hard sentences.",
          "outputs/analysis/complex_sentence_showdown/complex_sentence_showdown.json")

In [ ]:
if cs is not None:
    fields = list(cs["quartiles"])
    fig, axes = plt.subplots(1, len(fields), figsize=(5.8 * len(fields), 3.7), squeeze=False)
    for ax, field in zip(axes[0], fields):
        per = cs["quartiles"][field]
        qs = [q["quartile"] for q in list(per.values())[0]]
        xs = np.arange(len(qs))
        for i, (name, arr) in enumerate(per.items()):
            key = "spacy" if "spacy" in name else "llama" if "llama" in name else "gpt"
            ax.plot(xs, [pp(q["empty_rate"]) for q in arr], marker="osD^"[i % 4], ms=6, lw=2,
                    color=C[key], label=name,
                    ls="--" if key == "spacy" else "-")
        ax.set_xticks(xs); ax.set_xticklabels([q.split()[0] for q in qs])
        ax.set_xlabel("complexity quartile (Q1 simplest → Q4 hardest)")
        ax.set_ylabel("slot left empty (%)"); ax.set_title(f"`{field}`")
        ax.set_ylim(bottom=0); ax.grid(axis="x", visible=False)
    axes[0][0].legend(loc="best")
    fig.suptitle("Parser coverage across complexity quartiles — the gap is a level, not a slope",
                 y=1.02, fontsize=11.5, fontweight="bold")
    fig.tight_layout()
    fig_save(fig, "04_parser_quartiles",
             "Declined-slot rate by complexity quartile. spaCy is uniformly worse; it does "
             "not degrade faster (R3.1, R4.2).")

# Target accuracy figure
if ta:
    real = {k: v for k, v in ta.items() if k != "none"}
    if real:
        order = sorted(real, key=lambda k: -real[k]["splits"]["train"]["fuzzy_pct"])
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.7),
                                     gridspec_kw={"width_ratios": [1, 1.25]})
        xs = np.arange(len(order)); w = 0.26
        for j, (m, lbl) in enumerate([("exact_pct", "exact"), ("substring_pct", "substring"),
                                      ("fuzzy_pct", "fuzzy")]):
            a1.bar(xs + (j - 1) * w, [real[k]["splits"]["train"][m] for k in order], w,
                   label=lbl, hatch=HATCH[j], edgecolor="white",
                   color=["#9ecae1", "#4a98c9", "#0072B2"][j])
        a1.set_xticks(xs); a1.set_xticklabels(order, rotation=15)
        a1.set_ylabel("target accuracy (%)"); a1.set_title("Target field vs object_name")
        a1.legend(loc="lower left"); a1.set_ylim(0, 100); a1.grid(axis="x", visible=False)
        for k, x in zip(order, xs):
            v = real[k]["splits"]["train"]["fuzzy_pct"]
            a1.text(x + w, v + 1.5, f"{v:.1f}", ha="center", fontsize=8)

        errs = {}
        for k in order:
            for e in real[k]["splits"]["train"].get("top_errors", [])[:6]:
                errs.setdefault(k, []).append((f"{e['ground_truth']}→{e['predicted']}", e["count"]))
        base_k = order[0]
        pairs = errs.get(base_k, [])[:8]
        a2.barh([p[0] for p in pairs][::-1], [p[1] for p in pairs][::-1],
                color=C["gpt"], edgecolor="white")
        a2.set_xlabel("occurrences"); a2.set_title(f"Most frequent target errors — {base_k}")
        a2.grid(axis="y", visible=False)
        fig.tight_layout()
        fig_save(fig, "04_target_accuracy",
                 "Target-field accuracy by parser, and the dominant error type — almost all "
                 "synonym mismatches (R3.1).")

---
## 5 · Do parse errors hurt grounding? (R2, R4.3)

R2 asks for *"analysis of how errors in LLM sentence parsing propagate to the grounding
results."* There are two ways to answer, and they are not equivalent:

| | design | status |
|---|---|---|
| **correlational** | split annotations by whether the parse was right, compare accuracy | **available** |
| **causal** | hold the sample fixed, corrupt only the parse, re-evaluate | **missing** (needs GPU) |

The correlational version is confounded — badly-parsed descriptions skew toward rare classes
and unusual phrasing, which are harder to ground *regardless* of the parse. The
class-controlled (Cochran–Mantel–Haenszel) column removes that confound.

In [ ]:
pq = load_json("outputs/analysis/parse_quality_split/parse_quality_split.json")
if pq is None:
    skip("parse-quality-split", AVAIL["parse-quality-split"]["pattern"],
         "grounding accuracy vs parse correctness (R4.3)", AVAIL["parse-quality-split"]["blocker"])
else:
    both(f"**model** `{pq['model']}` · **split** `{pq['split']}` · "
         f"**criterion** `{pq['criterion']}` · **IoU** {pq['threshold']}")
    rows = []
    for name, s in pq["parsers"].items():
        cc, ww = s["correct"], s["wrong"]
        cmh = s.get("category_controlled", {})
        praw = s.get("p_value"); pc = cmh.get("p_value")
        if name != "none":
            track_p(f"parse-quality raw ({name})", praw, "parse_quality")
            track_p(f"parse-quality CMH ({name})", pc, "parse_quality")
        rows.append([f"**{name}**", f"{cc.get('n',0):,}", f"{pp(cc.get('acc')):.2f}" if cc.get('acc') is not None else "--",
                     f"{ww.get('n',0):,}", f"{pp(ww.get('acc')):.2f}" if ww.get('acc') is not None else "--",
                     f"{pp(s.get('raw_difference')):+.2f}" if s.get('raw_difference') is not None else "--",
                     f"{praw:.2e}" if praw is not None and not math.isnan(praw) else "--",
                     f"{pp(cmh.get('difference')):+.2f}" if cmh.get('difference') is not None else "--",
                     f"{pc:.2e}" if pc is not None and not math.isnan(pc) else "--",
                     stars(pc)])
    mdtable(["parser", "n correct", "Acc correct", "n wrong", "Acc wrong",
             "raw diff (pp)", "raw p", "controlled diff (pp)", "CMH p", "sig"],
            rows, name="05_parse_quality_split",
            caption="Grounding accuracy split by parse correctness, raw and class-controlled "
                    "(R4.3). The `none` row is degenerate by construction: every parse is 'wrong'.")
    NUMBERS["parse_quality_split"] = pq["parsers"]
    both("""
**Reading.** The raw difference is small and **not** significant for any parser. Controlling
for object class makes it larger *and* significant everywhere — the confound was masking the
effect, not creating it.

Neither column is an intervention — both compare *different annotations*. The causal version
is the corruption sweep in **§5b**, which holds the sample fixed and changes only the parse;
read the two together.
""")
    claim("safe", "After controlling for object class, grounding accuracy is significantly "
                  "higher on correctly-parsed annotations (CMH p < 0.01 for every parser).",
          "outputs/analysis/parse_quality_split/parse_quality_split.json")


In [ ]:
if pq is not None:
    names = [k for k in pq["parsers"] if k != "none"]
    if names:
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
        xs = np.arange(len(names)); w = 0.36
        cor = [pp(pq["parsers"][n]["correct"]["acc"]) for n in names]
        wro = [pp(pq["parsers"][n]["wrong"]["acc"]) for n in names]
        a1.bar(xs - w/2, cor, w, label="parse correct", color="#009E73", edgecolor="white")
        a1.bar(xs + w/2, wro, w, label="parse wrong", color=C["warn"], hatch="///",
               edgecolor="white")
        for x, (c, wv, n) in enumerate(zip(cor, wro, names)):
            a1.text(x - w/2, c + 0.6, f"{c:.1f}", ha="center", fontsize=8)
            a1.text(x + w/2, wv + 0.6, f"{wv:.1f}", ha="center", fontsize=8)
            a1.text(x, min(c, wv) - 4, f"n={pq['parsers'][n]['wrong']['n']:,}\nwrong",
                    ha="center", fontsize=7, color=C["ink"])
        a1.set_xticks(xs); a1.set_xticklabels(names); a1.set_ylabel("Acc@0.25 (%)")
        a1.set_title("Accuracy by parse correctness"); a1.legend(loc="lower right")
        a1.set_ylim(0, max(cor + wro) * 1.28); a1.grid(axis="x", visible=False)

        raw = [pp(pq["parsers"][n]["raw_difference"]) for n in names]
        ctl = [pp(pq["parsers"][n]["category_controlled"]["difference"]) for n in names]
        a2.bar(xs - w/2, raw, w, label="raw (confounded)", color="#BBBBBB", edgecolor="white")
        a2.bar(xs + w/2, ctl, w, label="class-controlled (CMH)", color=C["ours"],
               hatch="...", edgecolor="white")
        for x, n in enumerate(names):
            p = pq["parsers"][n]["category_controlled"].get("p_value")
            a2.text(x + w/2, ctl[x] + 0.15, stars(p), ha="center", fontsize=9)
        a2.axhline(0, color=C["ink"], lw=0.8)
        a2.set_xticks(xs); a2.set_xticklabels(names)
        a2.set_ylabel("accuracy difference (pp)")
        a2.set_title("Confounding removed by controlling for object class")
        a2.legend(loc="upper left"); a2.grid(axis="x", visible=False)
        fig.tight_layout()
        fig_save(fig, "05_parse_quality",
                 "Grounding accuracy vs parse correctness. Controlling for object class both "
                 "enlarges the effect and makes it significant (R4.3).")

---
## 5b · Parse errors, causally (R2, R4.3)

The corruption sweep has now run. This is the **intervention** that §5 could only approximate:
the sample is held fixed, only the parse is degraded, and the same trained model is
re-evaluated at each corruption level.

In [ ]:
epj = load_json("outputs/analysis/parse_error_propagation/parse_error_propagation.json")
if epj is None:
    skip("error-propagation", "outputs/analysis/parse_error_propagation/parse_error_propagation.json",
         "the causal parse-error -> grounding link (R2, R4.3)", "needs the corruption sweep")
else:
    both(f"**split** `{epj['split']}` · **IoU** {epj['threshold']} · "
         f"**source** `{epj['run_dir']}`")
    rows = []
    for L in epj["levels"]:
        pr = L.get("paired", {})
        p = pr.get("mcnemar_p")
        if p is not None:
            track_p(f"corruption {L['name']} (paired, corrupted subset)", p, "corruption")
        rows.append([f"**{L['name']}**", f"{pp(L['rate']):.0f}", f"{L['n']:,}",
                     f"{pp(L['acc']):.2f}",
                     f"{pp(L.get('corrupted_subset', {}).get('acc')):.2f}"
                     if L.get("corrupted_subset") else "--",
                     f"{pp(L.get('untouched_subset', {}).get('acc')):.2f}"
                     if L.get("untouched_subset") else "--",
                     f"{pr.get('broke','--')}/{pr.get('fixed','--')}" if pr else "--",
                     f"{p:.2e}" if p is not None else "--", stars(p) if p is not None else ""])
    mdtable(["level", "rate %", "n", "global Acc@0.25", "corrupted subset",
             "untouched (control)", "broke/fixed", "McNemar p", "sig"],
            rows, name="05b_error_propagation",
            caption="Causal parse-error propagation: same model, same annotations, only the "
                    "parse degraded (R2, R4.3).")
    NUMBERS["error_propagation"] = epj["levels"]
    for v in epj.get("verdicts", []):
        both(f"- {v}")

    base = epj["levels"][0]; worst = epj["levels"][-1]
    drop = pp(base["acc"]) - pp(worst["acc"])
    csub = worst.get("corrupted_subset", {}).get("acc")
    usub = worst.get("untouched_subset", {}).get("acc")
    both(f"""
**Reading, and it is a genuinely useful result.** Two features make this convincing where §5
was only suggestive:

1. **The control column holds.** Accuracy on *untouched* annotations stays flat
   (~{pp(usub):.1f}%) at every corruption level. Whatever moves is caused by the parse, not
   by drift in the evaluation.
2. **The paired test is on the corrupted annotations only**, comparing each one against its
   own baseline outcome. At 50% corruption {worst['paired']['broke']} annotations broke and
   {worst['paired']['fixed']} were accidentally fixed — a real, significant net harm.

**The effect is small in absolute terms**: corrupting *half* of all parses costs
**{drop:.2f} pp** globally, because the untouched majority dilutes it. On the corrupted
subset itself the gap is larger ({pp(usub):.1f}% → {pp(csub):.1f}%).

This is the sentence for the response letter: *parse errors propagate to grounding
measurably and significantly, but the model is fairly robust to them* — which is a more
defensible and more interesting claim than either extreme.
""")
    claim("safe",
          f"Parse errors propagate causally: corrupting 50% of parses costs {drop:.2f} pp "
          f"globally, with a significant paired effect on the corrupted subset "
          f"(McNemar p={worst['paired']['mcnemar_p']:.1e}) and a flat untouched control.",
          "outputs/analysis/parse_error_propagation/parse_error_propagation.json")

In [ ]:
if epj is not None:
    L = epj["levels"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.9))
    rates = [pp(x["rate"]) for x in L]
    glob_acc = [pp(x["acc"]) for x in L]
    lo = [pp(x["acc"]) - pp(x["ci"][0]) for x in L]
    hi = [pp(x["ci"][1]) - pp(x["acc"]) for x in L]
    a1.errorbar(rates, glob_acc, yerr=[lo, hi], marker="o", ms=7, lw=2, capsize=4,
                color=C["ours"], label="global (all annotations)")
    csub = [pp(x["corrupted_subset"]["acc"]) if x.get("corrupted_subset") else None for x in L]
    usub = [pp(x["untouched_subset"]["acc"]) if x.get("untouched_subset") else None for x in L]
    ok = [i for i, v in enumerate(csub) if v is not None]
    a1.plot([rates[i] for i in ok], [csub[i] for i in ok], marker="D", ms=7, lw=2,
            color=C["base"], ls="--", label="corrupted subset only")
    a1.plot([rates[i] for i in ok], [usub[i] for i in ok], marker="s", ms=6, lw=2,
            color="#009E73", ls=":", label="untouched subset (control)")
    a1.set_xlabel("parse corruption rate (%)"); a1.set_ylabel("Acc@0.25 (%)")
    a1.set_title("Dose–response: degrade only the parse")
    a1.legend(loc="lower left", fontsize=8.5); a1.grid(axis="x", visible=False)

    pr = [(x["name"], x["paired"]) for x in L if x.get("paired")]
    if pr:
        xs = np.arange(len(pr)); w = 0.36
        a2.bar(xs - w/2, [p["broke"] for _, p in pr], w, label="broke (correct→wrong)",
               color=C["base"], edgecolor="white")
        a2.bar(xs + w/2, [p["fixed"] for _, p in pr], w, label="fixed (wrong→correct)",
               color="#009E73", hatch="///", edgecolor="white")
        for x, (n, p) in zip(xs, pr):
            a2.text(x, max(p["broke"], p["fixed"]) + 4,
                    stars(p.get("mcnemar_p")), ha="center", fontsize=10)
        a2.set_xticks(xs); a2.set_xticklabels([n for n, _ in pr])
        a2.set_ylabel("annotations that flipped")
        a2.set_title("Paired flips on the corrupted subset")
        a2.legend(loc="upper left", fontsize=8.5); a2.grid(axis="x", visible=False)
    fig.tight_layout()
    fig_save(fig, "05b_error_propagation",
             "Causal parse-error propagation. The flat control line is what makes the "
             "dose–response interpretable (R2, R4.3).")

---
## 5c · Train-time vs test-time parser swap (R4.4)

Two different questions, and the repository now has both:

- **train-time** (§2b) — retrain with a different parser. Answers *"does the parser matter to
  the method?"*
- **test-time** (here) — take one trained model and feed it another parser's output at
  inference. Answers *"how sensitive is a deployed model to its parser?"*

Comparing them is more informative than either alone, because the difference between the two
is exactly the part the model can *learn to absorb*.

In [ ]:
SWAP_DIR = REPO / "outputs" / CHECKPOINT / "parser_swap"
swap = {}
if SWAP_DIR.is_dir():
    for d in sorted(SWAP_DIR.iterdir()):
        f = d / "eval_stdout.txt"
        if not f.is_file(): continue
        t = f.read_text(errors="ignore")
        m25 = re.findall(r"overall \| overall \| acc@0\.25iou:\s*([0-9.]+)", t)
        m50 = re.findall(r"overall \| overall \| acc@0\.5iou:\s*([0-9.]+)", t)
        if m25:
            swap[d.name] = {"acc_0.25": float(m25[-1]),
                            "acc_0.5": float(m50[-1]) if m50 else None}

if not swap:
    skip("eval-only-swap", f"outputs/{CHECKPOINT}/parser_swap/*/eval_stdout.txt",
         "test-time parser sensitivity (R4.4)", "the swap evaluation has not run")
else:
    ref_swap = swap.get("gpt4o-mini", {}).get("acc_0.25")
    rows = []
    for name in sorted(swap, key=lambda k: -swap[k]["acc_0.25"]):
        v = swap[name]
        d = (v["acc_0.25"] - ref_swap) if ref_swap else None
        rows.append([f"**{name}**", f"{pp(v['acc_0.25']):.2f}",
                     f"{pp(v['acc_0.5']):.2f}" if v["acc_0.5"] else "--",
                     f"{pp(d):+.2f}" if d is not None else "--"])
    mdtable(["parser fed at test time", "Acc@0.25", "Acc@0.5", "Δ vs GPT (pp)"],
            rows, name="05c_parser_swap",
            caption="Evaluation-only parser swap: one trained model, five different parse "
                    "caches at inference (R4.4).")
    NUMBERS["parser_swap"] = swap
    both("This is a **sensitivity** analysis: the model was trained on GPT parses, so part of "
         "each drop is distribution mismatch rather than parse quality.")

    # ---- the cross-comparison ----
    if RUNS and ref_mean is not None and ref_swap:
        pmap = {"spacy": "ABL-PARSER-SPACY", "none": "ABL-PARSER-NONE",
                "llama": "ABL-PARSER-LLAMA", "smalllm": "ABL-PARSER-SMALLLM"}
        rows, pairs = [], []
        for lab, arm in pmap.items():
            if lab not in swap: continue
            dt = pp(swap[lab]["acc_0.25"] - ref_swap)
            dr = pp(RUNS[arm]["acc_0.25"] - ref_mean) if arm in RUNS else None
            rows.append([f"**{lab}**",
                         f"{pp(swap[lab]['acc_0.25']):.2f}", f"{dt:+.2f}",
                         f"{pp(RUNS[arm]['acc_0.25']):.2f}" if arm in RUNS else "not run",
                         f"{dr:+.2f}" if dr is not None else "--",
                         f"{dr - dt:+.2f}" if dr is not None else "--"])
            if dr is not None: pairs.append((lab, dt, dr))
        mdtable(["parser", "test-time Acc", "Δ test-time", "train-time Acc", "Δ train-time",
                 "retraining recovers"], rows, name="05c_train_vs_test",
                caption="The same parser substituted at test time versus trained with. The "
                        "last column is how much retraining recovers (R4.4).")
        if pairs:
            both(f"""
**The most informative comparison in this notebook.** For each parser, the test-time drop is
what a *deployed* model suffers; the train-time drop is what the *method* suffers once it has
adapted. The gap between them is the part retraining absorbs.

{chr(10).join(f"- **{l}**: test-time {dt:+.2f} pp → train-time {dr:+.2f} pp "
              f"({'retraining recovers ' + format(dr - dt, '+.2f') + ' pp' if dr > dt else 'retraining does not help'})"
              for l, dt, dr in pairs)}

Where retraining fails to recover the gap, the parser supplies information the architecture
genuinely needs; where it recovers most of it, the model had simply met an unfamiliar input
distribution.
""")


In [ ]:
if swap:
    fig, ax = plt.subplots(figsize=(8.6, 4.0))
    labs = [l for l in ("gpt4o-mini", "llama", "spacy", "smalllm", "none") if l in swap]
    xs = np.arange(len(labs)); w = 0.38
    tv = [pp(swap[l]["acc_0.25"]) for l in labs]
    ax.bar(xs - w/2, tv, w, label="test-time swap (frozen model)", color=C["base"],
           edgecolor="white")
    pmap = {"spacy": "ABL-PARSER-SPACY", "none": "ABL-PARSER-NONE",
            "llama": "ABL-PARSER-LLAMA", "smalllm": "ABL-PARSER-SMALLLM",
            "gpt4o-mini": None}
    rv, rx = [], []
    for i, l in enumerate(labs):
        arm = pmap.get(l)
        v = (ref_mean if l == "gpt4o-mini" else
             (RUNS[arm]["acc_0.25"] if arm in RUNS else None)) if RUNS else None
        if v is not None:
            rv.append(pp(v)); rx.append(xs[i] + w/2)
    if rv:
        ax.bar(rx, rv, w, label="retrained with that parser", color=C["ours"],
               hatch="...", edgecolor="white")
    for x, v in zip(xs - w/2, tv):
        ax.text(x, v + 0.06, f"{v:.2f}", ha="center", fontsize=8)
    for x, v in zip(rx, rv):
        ax.text(x, v + 0.06, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(xs); ax.set_xticklabels(labs)
    ax.set_ylabel("Acc@0.25 (%)")
    ax.set_ylim(min(tv + rv) - 0.7, max(tv + rv) + 0.5)
    ax.set_title("Parser substituted at test time vs trained with (R4.4)")
    ax.legend(loc="lower left", fontsize=8.5); ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig_save(fig, "05c_train_vs_test",
             "Test-time parser swap against full retraining. The gap between the pairs is "
             "what retraining absorbs (R4.4).")

---
## 6 · Failure analysis (R2, R4)

R2: *"The qualitative analysis only highlights successful examples. Including failure cases
and discussing situations where the model fails would provide a more balanced evaluation."*

In [ ]:
fc = load_json("outputs/analysis/failure_cases/failure_cases.json")
if fc is None:
    skip("failure-cases", AVAIL["failure-cases"]["pattern"], "failure taxonomy (R2, R4)",
         AVAIL["failure-cases"]["blocker"])
else:
    nf, na = fc["num_failures"], fc["num_annotations"]
    both(f"**{nf:,} of {na:,} annotations ({nf/na*100:.2f}%) fail** at IoU < {fc['threshold']} "
         f"· model `{fc['model']}` · criterion `{fc['criterion']}`")
    dist = fc["cause_distribution"]
    tot = sum(dist.values())
    rows = []
    for cause, cnt in sorted(dist.items(), key=lambda kv: -kv[1]):
        lo, hi = wilson(cnt, tot)
        rows.append([f"**{cause}**", f"{cnt:,}", f"{cnt/tot*100:.1f}",
                     f"[{pp(lo):.1f}–{pp(hi):.1f}]", f"{cnt/na*100:.1f}"])
    mdtable(["cause", "count", "% of failures", "95% CI", "% of all annotations"],
            rows, name="06_failure_causes",
            caption="Failure cause distribution with Wilson CIs (R2, R4).")
    NUMBERS["failure_causes"] = dist
    NUMBERS["failure_summary"] = dict(num_failures=nf, num_annotations=na,
                                      failure_rate=nf/na)
    top = max(dist, key=dist.get)
    parse_share = dist.get("parse_target_wrong", 0) / tot * 100
    both(f"""
**Reading — and this cuts against the paper's own framing.** The dominant failure is
**`{top}` at {dist[top]/tot*100:.1f}%** of failures: the right *class* is found and the wrong
*instance* is selected. Meanwhile **parse errors account for only {parse_share:.1f}%**.

Two uses for this. Defensively, it answers reviewers who imply the parser is the weak link —
it is not, by a factor of ten. But it also invites the question of whether a
parsing-centric framing matches the model's actual error profile, where the bottleneck is
instance discrimination among same-class distractors. Better to raise this in the response
letter than to let a reviewer raise it.
""")
    claim("safe", f"{dist[top]/tot*100:.1f}% of failures are same-class instance confusion, "
                  f"while parse errors explain only {parse_share:.1f}%.",
          "outputs/analysis/failure_cases/failure_cases.json")

    sel = fc.get("selected", [])
    if sel:
        h("6.1 · Inspected cases", 5)
        rows = [[c["cause"], f"{c['iou']:.3f}", c["object_name"],
                 f"`{c['scene_id']}`/{c['object_id']}",
                 (c["description"][:66] + "…") if len(c["description"]) > 66 else c["description"]]
                for c in sel]
        mdtable(["cause", "IoU", "class", "scene/obj", "description"], rows,
                name="06_failure_cases", caption="Individually inspected failure cases.")
        figs = load_json("experiments/analysis/figures/figures.json")
        note(f"Rendered figures: `experiments/analysis/figures/` "
             f"({'available' if figs else 'not found'}); per-case `pred.ply`/`gt.ply` under "
             f"`outputs/analysis/failure_cases/`.")

et = load_json("outputs/analysis/annotation/error_taxonomy.json")
if et is None:
    skip("error-taxonomy", AVAIL["error-taxonomy"]["pattern"],
         "manual parse-error rates on a labelled subset (R1.2, R4.3)",
         AVAIL["error-taxonomy"]["blocker"])
else:
    h("6.2 · Manual parse assessment (R1.2, R4.3)", 4)
    both("R4.3 asks for *'a manual assessment on a representative subset'*. The sample "
         "**deliberately oversamples target failures**, so the *population* column — "
         "re-weighted by stratum — is the one to quote as a dataset rate.")
    rows = []
    for name, blk in et["parsers"].items():
        r = [f"**{name}**", blk.get("labelled_rows", blk.get("n_rows"))]
        for f in ("target_ok", "adjectives_ok", "neighbors_ok"):
            d = blk["fields"].get(f, {})
            s, p = d.get("sample_pct"), d.get("population_pct")
            r.append(f"{s:.1f} / **{p:.1f}**" if s is not None and p is not None else "--")
        rows.append(r)
    mdtable(["parser", "n labelled", "target ok (sample/pop %)",
             "adjectives ok", "neighbors ok"], rows, name="06_error_taxonomy",
            caption="Manual parse assessment, sample and population-weighted rates (R1.2, R4.3).")
    codes = {k: v for k, v in list(et["parsers"].values())[0].get("error_codes", {}).items()}
    if codes:
        rows = [[f"`{k}`", v, f"{v/max(1,list(et['parsers'].values())[0].get('labelled_rows',1))*100:.1f}"]
                for k, v in sorted(codes.items(), key=lambda kv: -kv[1])]
        mdtable(["error code", "count", "% of labelled rows"], rows,
                name="06_error_codes", caption="Manual error taxonomy.")
    NUMBERS["error_taxonomy"] = et["parsers"]
    labelled = [k for k in et["parsers"]]
    if len(labelled) < 4:
        both(f"""
> **Incomplete, and unblocked.** Sheets were generated for **4 parsers** but only
> **{len(labelled)}** ({', '.join(labelled)}) has been manually labelled, at
> n={list(et['parsers'].values())[0].get('labelled_rows','?')}. This needs **human labelling, not
> compute** — the highest-value task in the repository that no hardware is blocking.
> n=30 is also thin for a per-field percentage; the CI is wide.
""")

In [ ]:
if fc is not None:
    dist = fc["cause_distribution"]; tot = sum(dist.values())
    order = sorted(dist, key=lambda k: -dist[k])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.9),
                                 gridspec_kw={"width_ratios": [1.35, 1]})
    vals = [dist[k] / tot * 100 for k in order]
    cols = [C["warn"] if k == "parse_target_wrong" else
            C["base"] if k == order[0] else C["ours"] for k in order]
    bars = a1.barh(range(len(order)), vals, color=cols, edgecolor="white")
    for i, (k, v) in enumerate(zip(order, vals)):
        lo, hi = wilson(dist[k], tot)
        a1.plot([pp(lo), pp(hi)], [i, i], color=C["ink"], lw=1.2)
        a1.text(v + 2.2, i, f"{v:.1f}%  (n={dist[k]:,})", va="center", fontsize=8.5)
    a1.set_yticks(range(len(order))); a1.set_yticklabels(order, fontsize=9)
    a1.invert_yaxis(); a1.set_xlabel("% of failures (Wilson 95% CI)")
    a1.set_title(f"Failure causes — {fc['num_failures']:,} failures "
                 f"({fc['num_failures']/fc['num_annotations']*100:.1f}% of the split)")
    a1.set_xlim(0, max(vals) * 1.42); a1.grid(axis="y", visible=False)

    # success vs failure, with the parse share isolated
    succ = fc["num_annotations"] - fc["num_failures"]
    pw = dist.get("parse_target_wrong", 0)
    a2.bar(["all annotations"], [succ], color="#009E73", label="grounded", edgecolor="white")
    a2.bar(["all annotations"], [fc["num_failures"] - pw], bottom=[succ], color="#BBBBBB",
           label="failed, not parse", edgecolor="white")
    a2.bar(["all annotations"], [pw], bottom=[succ + fc["num_failures"] - pw],
           color=C["warn"], hatch="///", label="failed, parse error", edgecolor="white")
    a2.set_ylabel("annotations"); a2.set_title("Parse errors are a small slice")
    a2.legend(loc="lower center", bbox_to_anchor=(0.5, -0.42))
    a2.text(0, succ + fc["num_failures"] * 0.5, f"parse-caused:\n{pw/fc['num_annotations']*100:.1f}% of all",
            ha="center", fontsize=8.5, color=C["ink"])
    a2.grid(axis="x", visible=False)
    fig.tight_layout()
    fig_save(fig, "06_failures",
             "Failure taxonomy. Same-class distractor confusion dominates; parse errors are "
             "a small minority (R2, R4).")

---
## 7 · Equation 7 — is the sign right? (R1.3, R4.6)

Both reviewers challenge the same thing. R4.6: *"Equation 7 adds the distance matrix to the
attention logits, but its normalization and sign are not defined. If positive distances are
used directly, farther proposals would receive larger attention biases, contrary to the
stated intention."*

This is a factual question with a measurable answer.

In [ ]:
eq = load_json("outputs/diagnostics/eq7_distance_bias.json")
if eq is None:
    skip("verify-eq7", AVAIL["verify-eq7"]["pattern"],
         "sign and normalisation of Eq. 7 (R1.3, R4.6)", AVAIL["verify-eq7"]["blocker"])
else:
    both(f"**combination** `{eq['way']}` · **proposals** {eq['num_proposals']} · "
         f"**logit σ** {eq['logit_std']:.4f}")
    rows = []
    for hd, d in sorted(eq["per_head"].items()):
        inert = int(hd) in eq.get("inert_heads", [])
        rows.append([f"head {hd}" + (" *(inert)*" if inert else " **(active)**"),
                     f"{eq['channel_std'][int(hd)]:.4f}",
                     f"{d['rho']:+.4f}", f"{d['ratio']:.2f}",
                     f"{d['rho_control']:+.4f}", f"{d['ratio_control']:.3f}"])
    mdtable(["head", "channel σ", "ρ(distance, attention)", "near/far ratio",
             "ρ control", "ratio control"], rows, name="07_eq7",
            caption="Eq. 7 distance bias per head. ρ < 0 means nearer proposals receive more "
                    "attention — the intended behaviour (R1.3, R4.6).")
    NUMBERS["eq7"] = eq

    act = [hd for hd in eq["per_head"] if int(hd) not in eq.get("inert_heads", [])]
    if act:
        a = act[0]; d = eq["per_head"][a]
        both(f"""
**The reviewers' concern does not hold, and here is the number.** On the only active head
(head {a}), ρ(distance, attention) = **{d['rho']:+.4f}** with a near/far attention ratio of
**{d['ratio']:.1f}×**. The correlation is strongly *negative*: **nearer proposals receive more
attention**, which is the stated intention. Under the control condition the effect vanishes
(ρ = {d['rho_control']:+.4f}, ratio {d['ratio_control']:.3f}), confirming the distance term is
doing the work rather than some confound.
""")
        claim("safe", f"Eq. 7's sign is correct: on the active head ρ(distance, attention) = "
                      f"{d['rho']:+.4f} ({d['ratio']:.1f}× near/far), and the effect disappears "
                      f"under control.",
              "outputs/diagnostics/eq7_distance_bias.json")
    inert = eq.get("inert_heads", [])
    if inert:
        both(f"""
> **An unprompted finding that should be reported anyway.** Heads {inert} are **inert** —
> their channel σ is ~0, meaning {len(inert)} of {len(eq['per_head'])} distance heads learned
> nothing at all. No reviewer asked, and it is mildly unflattering, but a reader reproducing
> the work would find it immediately. Disclosing it costs little and buys credibility; it
> also suggests the distance mechanism could be reduced to a single head with no loss.
""")
        claim("safe", f"{len(inert)} of {len(eq['per_head'])} distance-bias heads are inert "
                      f"(channel σ ≈ 0), so the mechanism effectively uses one head.",
              "outputs/diagnostics/eq7_distance_bias.json")

In [ ]:
if eq is not None:
    heads = sorted(eq["per_head"], key=int)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.7))
    xs = np.arange(len(heads)); w = 0.36
    rho = [eq["per_head"][h]["rho"] for h in heads]
    rhoc = [eq["per_head"][h]["rho_control"] for h in heads]
    a1.bar(xs - w/2, rho, w, label="measured", color=C["ours"], edgecolor="white")
    a1.bar(xs + w/2, rhoc, w, label="control", color="#BBBBBB", hatch="///", edgecolor="white")
    a1.axhline(0, color=C["ink"], lw=0.9)
    a1.set_xticks(xs)
    a1.set_xticklabels([f"head {h}" + ("\n(inert)" if int(h) in eq.get("inert_heads", []) else "\n(active)")
                        for h in heads], fontsize=8.5)
    a1.set_ylabel("ρ(distance, attention)")
    a1.set_title("Negative ρ = nearer proposals attend more  ✓")
    a1.legend(loc="lower left"); a1.grid(axis="x", visible=False)
    # Label inside the bar: a negative bar's "below" is where the tick labels live.
    for x, v in zip(xs - w/2, rho):
        if abs(v) > 0.05:
            a1.text(x, v * 0.92, f"{v:+.3f}", ha="center", va="bottom" if v < 0 else "top",
                    fontsize=8.5, fontweight="bold", color="white")
    a1.margins(y=0.12)

    sig = eq["channel_std"]
    cols = [C["base"] if s > 0.5 else "#DDDDDD" for s in sig]
    a2.bar(range(len(sig)), sig, color=cols, edgecolor="white")
    a2.set_xticks(range(len(sig))); a2.set_xticklabels([f"head {i}" for i in range(len(sig))],
                                                       fontsize=8.5)
    a2.set_ylabel("channel σ"); a2.set_title("Only one distance head is active")
    a2.grid(axis="x", visible=False)
    for i, s in enumerate(sig):
        a2.text(i, s + 0.03, f"{s:.3f}", ha="center", fontsize=8.5)
    fig.tight_layout()
    fig_save(fig, "07_eq7",
             "Eq. 7 verification. The sign is correct on the active head, and three of four "
             "heads are inert (R1.3, R4.6).")

---
## 8 · Efficiency and cost (R2, R4.5)

R2 asks for *"FLOPs, GPU memory consumption, and inference latency in addition to parameter
counts."* R4.5 asks whether parsing happens offline and what it costs per query.

In [ ]:
cx = load_json("outputs/complexity/complexity_report.json")
if cx is None:
    skip("complexity-model", AVAIL["complexity-model"]["pattern"],
         "FLOPs, peak memory and latency (R2, R4.5)", AVAIL["complexity-model"]["blocker"])
else:
    env = cx.get("environment", {})
    both(f"**device** `{env.get('device')}` · **torch** `{env.get('torch')}` · "
         f"**pointnet impl** `{env.get('pointnet_impl')}` · "
         f"**CUDA ext** {env.get('pointnet2_ops_cuda_ext')}")
    got, gaps = [], []
    for variant, r in cx.get("results", {}).items():
        prm = r.get("params", {})
        row = [f"**{variant}**",
               f"{prm.get('total', 0):,}", f"{prm.get('lang', 0):,}", f"{prm.get('match', 0):,}"]
        for k, lbl in [("flops", "FLOPs"), ("peak_memory_mb", "peak MB"), ("latency_bs1", "latency")]:
            v = r.get(k)
            row.append(f"{v:,.0f}" if isinstance(v, (int, float)) else "**null**")
            if v is None: gaps.append((variant, lbl))
        got.append(row)
    mdtable(["variant", "params total", "params lang", "params match", "FLOPs",
             "peak MB", "latency bs1"], got, name="08_complexity",
            caption="Model complexity (R2). Parameter counts are real; the rest is missing.")
    NUMBERS["complexity"] = cx
    err = list(cx.get("results", {}).values())[0].get("error")
    if gaps:
        both(f"""
> **Partially broken — parameter counts only.** FLOPs, peak memory and latency are all
> `null`. Recorded cause:
>
> ```
> {err}
> ```
>
> `MatchModule` calls `torch_geometric.nn.knn_graph` in its forward pass, so the profiler
> cannot trace the model without a working `pyg-lib`. **R2's request is therefore currently
> unanswered beyond parameter counts.** Two further staleness notes: the file records
> `pointnet_impl: {env.get('pointnet_impl')!r}`, a module path that no longer exists after the
> PointNet reorganisation, and `torch {env.get('torch')}`, which is not the currently installed
> build. Re-run `complexity-model` on a machine with a matching `pyg-lib` before quoting it.
""")
        claim("unsupported", "The paper reports FLOPs, GPU memory and inference latency.",
              "Only parameter counts exist; the rest is null from a pyg-lib import failure.")
    else:
        claim("safe", "FLOPs, peak memory and latency are measured.", "outputs/complexity/complexity_report.json")

pl = load_json("outputs/complexity/parsing_latency_report.json")
off = load_json("outputs/reports/parsing-offline-check/result.json")
if pl is None:
    skip("parsing-offline-check", AVAIL["parsing-offline-check"]["pattern"],
         "whether parsing is offline, and its per-query cost (R4.5)", "--")
else:
    h("8.1 · Is parsing in the inference path? (R4.5)", 4)
    dep = pl.get("deployment", {})
    verdict = dep.get("verdict", "")
    both(f"""
**Verdict: `{verdict}`**

- `lib/dataset.py` loads a precomputed tokenized parse JSON at construction:
  **{dep.get('dataset_loads_precomputed_json')}**
- parse cache read at construction: **{dep.get('parse_cache_read_at_construction')}**
- **parser invocations in the forward path: `{dep.get('parser_invocations_in_forward_path')}`**
- parse caches available: {len(dep.get('available_parse_caches', []))}

This answers R4.5 exactly as asked: no parser runs per query at inference time, so the
latency in Figure 11 is not missing a hidden GPT call.
""")
    claim("safe", "Parsing is fully offline: no parser is invoked in the forward path, so "
                  "reported inference latency is complete.",
          "outputs/complexity/parsing_latency_report.json")
    NUMBERS["parsing_offline"] = dep
    if pl.get("config", {}).get("mode") == "offline_check":
        lat = REPO / "outputs/reports/parsing-latency-spacy/log.txt"
        both(f"""
> **Stale-artifact warning.** `parsing_latency_report.json` currently holds
> `mode: offline_check` — the later offline check **overwrote** the spaCy latency run's
> output. The per-query spaCy numbers survive only in
> `{lat.relative_to(REPO) if lat.is_file() else 'outputs/reports/parsing-latency-spacy/log.txt'}`
> {'(present)' if lat.is_file() else '(missing)'}. Re-run `parsing-latency-spacy` if you need
> the figure for the paper; it takes ~10 s on CPU.
""")

au = load_json("outputs/diagnostics/scene_cache_audit_val.json")
if au is not None:
    h("8.2 · Detector ceiling — where is the bottleneck?", 4)
    ceil, ach = au.get("ceiling", {}), au.get("achieved", {})
    rows = [[f"@{t}", f"{pp(ceil.get(t)):.2f}", f"{pp(ach.get(t)):.2f}",
             f"{pp(ceil.get(t)) - pp(ach.get(t)):.2f}",
             f"{ach.get(t)/ceil.get(t)*100:.1f}"]
            for t in ("0.25", "0.5") if t in ceil and t in ach]
    mdtable(["IoU", "detector ceiling %", "achieved %", "headroom (pp)", "% of ceiling reached"],
            rows, name="08_ceiling",
            caption="Recall ceiling of the frozen detector vs achieved grounding accuracy.")
    NUMBERS["ceiling"] = dict(ceiling=ceil, achieved=ach,
                              integrity_ok=au.get("integrity_ok"))
    both(f"""
**Reading.** The detector can reach **{pp(ceil.get('0.25')):.1f}%** @0.25 but the full model
achieves **{pp(ach.get('0.25')):.1f}%** — only {ach.get('0.25')/ceil.get('0.25')*100:.0f}% of what
detection makes available. **The bottleneck is grounding, not detection.** That is a useful
framing for R3.2/R4.9: gains should be sought in the language–vision matching, and it also
explains why the frozen-detector ablation protocol is legitimate.
""")
    claim("safe", f"The frozen detector's recall ceiling is {pp(ceil.get('0.25')):.1f}% @0.25 while "
                  f"the model reaches {pp(ach.get('0.25')):.1f}%, so grounding — not detection — "
                  f"is the bottleneck.",
          "outputs/diagnostics/scene_cache_audit_val.json")
    if au.get("achieved_above_ceiling"):
        both(f"""
> **Reconcile before citing.** The file reports
> `achieved_above_ceiling: {au['achieved_above_ceiling']:,}` alongside `problems: []`. Those
> two look contradictory — {au['achieved_above_ceiling']:,} annotations scored above what the
> audit calls the ceiling. Understand which definition of "ceiling" is in play before this
> number goes in the paper.
""")

In [ ]:
if cx is not None or au is not None:
    ncols = int(cx is not None) + int(au is not None)
    fig, axes = plt.subplots(1, ncols, figsize=(5.7 * ncols, 3.6), squeeze=False)
    i = 0
    if cx is not None:
        ax = axes[0][i]; i += 1
        r = list(cx.get("results", {}).values())[0]
        prm = r.get("params", {})
        parts = {k: v for k, v in prm.items() if k != "total"}
        other = prm.get("total", 0) - sum(parts.values())
        if other > 0: parts["other"] = other
        ks = list(parts)
        ax.bar(ks, [parts[k] / 1e6 for k in ks],
               color=[C["ours"], C["base"], "#BBBBBB"][:len(ks)], edgecolor="white")
        for j, k in enumerate(ks):
            ax.text(j, parts[k] / 1e6 + 0.02, f"{parts[k]/1e6:.2f}M", ha="center", fontsize=8.5)
        ax.set_ylabel("parameters (millions)")
        ax.set_title(f"Parameters — {prm.get('total', 0):,} total")
        ax.grid(axis="x", visible=False)
        miss = [k for k in ("flops", "peak_memory_mb", "latency_bs1") if r.get(k) is None]
        if miss:
            ax.text(0.5, 0.55, "FLOPs / memory / latency\nNOT MEASURED\n(pyg-lib import failure)",
                    transform=ax.transAxes, ha="center", va="center", fontsize=9.5,
                    color=C["base"], fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.5", fc="#FFF3E6", ec=C["base"]))
    if au is not None:
        ax = axes[0][i]
        ts = [t for t in ("0.25", "0.5") if t in au.get("ceiling", {})]
        xs = np.arange(len(ts)); w = 0.36
        ax.bar(xs - w/2, [pp(au["ceiling"][t]) for t in ts], w, label="detector ceiling",
               color="#BBBBBB", edgecolor="white")
        ax.bar(xs + w/2, [pp(au["achieved"][t]) for t in ts], w, label="achieved",
               color=C["ours"], hatch="...", edgecolor="white")
        for x, t in zip(xs, ts):
            c, a = pp(au["ceiling"][t]), pp(au["achieved"][t])
            ax.annotate("", xy=(x + w/2, a), xytext=(x - w/2, c),
                        arrowprops=dict(arrowstyle="<->", color=C["base"], lw=1.3))
            ax.text(x, (c + a) / 2, f"  {c-a:.1f} pp\n  headroom", fontsize=8, color=C["base"])
        ax.set_xticks(xs); ax.set_xticklabels([f"IoU {t}" for t in ts])
        ax.set_ylabel("accuracy (%)"); ax.set_title("Grounding is the bottleneck, not detection")
        ax.legend(loc="upper right"); ax.grid(axis="x", visible=False)
    fig.tight_layout()
    fig_save(fig, "08_efficiency",
             "Parameter breakdown (FLOPs/latency still unmeasured) and the detector recall "
             "ceiling versus achieved accuracy (R2).")

---
## 9 · Multiple comparisons

This notebook has now collected many p-values across several independent analyses. Reporting
each at α = 0.05 inflates the family-wise error rate — if a reviewer counts the tests, the
weakest claims are the ones that will be challenged first.

**Holm–Bonferroni** is applied below: uniformly more powerful than plain Bonferroni, and it
makes no independence assumption. Corrections are computed *within* each family, since the
families test different hypotheses.

In [ ]:
if not PVALUES:
    note("> No p-values were collected — the analyses that produce them were all skipped.")
else:
    fams = sorted({f for _, _, f in PVALUES})
    NUMBERS["multiple_comparisons"] = {}
    for fam in fams:
        pairs = [(l, p) for l, p, f in PVALUES if f == fam]
        res = holm(pairs)
        h(f"family `{fam}` — {len(res)} tests", 4)
        rows = [[l, f"{p:.3e}", f"{adj:.3e}", "**reject H₀**" if rej else "retain H₀",
                 stars(adj)] for l, p, adj, rej in res]
        mdtable(["comparison", "raw p", "Holm-adjusted p", "decision at α=0.05", "sig"],
                rows, name=f"09_holm_{fam}",
                caption=f"Holm–Bonferroni correction within family `{fam}`.")
        nk = sum(1 for *_, rej in res if rej)
        both(f"- **{nk} of {len(res)}** comparisons in `{fam}` survive correction.")
        NUMBERS["multiple_comparisons"][fam] = [
            dict(label=l, p=p, p_adj=adj, reject=rej) for l, p, adj, rej in res]

    all_res = holm([(l, p) for l, p, _ in PVALUES])
    surv = sum(1 for *_, r in all_res if r)
    both(f"""
**Across every family pooled** ({len(all_res)} tests), **{surv}** remain significant after
Holm correction. Report corrected p-values for any claim the paper leans on; the per-bin
complexity gaps are the ones most exposed, since they are the largest family of tests
against the smallest effects.
""")

In [ ]:
if PVALUES:
    all_res = holm([(l, p) for l, p, _ in PVALUES])
    fig, ax = plt.subplots(figsize=(9.2, max(3.2, 0.32 * len(all_res))))
    ys = np.arange(len(all_res))
    raw = [-math.log10(max(p, 1e-300)) for _, p, _, _ in all_res]
    adj = [-math.log10(max(a, 1e-300)) for _, _, a, _ in all_res]
    ax.barh(ys, raw, 0.42, label="raw p", color="#BBBBBB", edgecolor="white")
    ax.barh(ys + 0.42, adj, 0.42, label="Holm-adjusted", color=C["ours"], edgecolor="white")
    ax.axvline(-math.log10(0.05), color=C["base"], ls="--", lw=1.4, label="α = 0.05")
    ax.set_yticks(ys + 0.21)
    ax.set_yticklabels([l if len(l) < 46 else l[:43] + "…" for l, *_ in all_res], fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel("−log₁₀(p)   (right of the dashed line = significant)")
    ax.set_title("Every comparison, before and after multiple-comparison correction")
    ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
    fig.tight_layout()
    fig_save(fig, "09_multiple_comparisons",
             "All collected p-values, raw and Holm-adjusted. Bars right of the dashed line "
             "survive correction.")

---
## 10 · Reviewer coverage

Every numbered concern from the four reviewers, mapped to the evidence that exists on disk
right now. This is the table to work from when drafting the point-by-point response.

In [ ]:
# (reviewer point, text, status, evidence / blocker)
def st(cond_full, cond_part=False):
    return "COMPLETE" if cond_full else ("PARTIAL" if cond_part else "MISSING")

have = lambda k: AVAIL.get(k, {}).get("ok", False)
trained = any(have(k) for k in
              ["train-parser-gpt","train-parser-spacy","train-parser-llama","train-parser-none"])

REVIEW = [
 ("R1.1","expand related work (2D/3D survey, monocular)","WRITING","manuscript only, no experiment"),
 ("R1.2","more parsing examples, success and failure","COMPLETE" if (have("complex-sentence-showdown") and have("failure-cases")) else "PARTIAL",
  "§4.3 case tables · §6 failure taxonomy · §6.2 manual sheets"),
 ("R1.3","clarify distance matrix construction / sign","COMPLETE" if have("verify-eq7") else "MISSING",
  "§7 — rho measured, sign confirmed correct"),
 ("R1.4","discuss fairness of comparisons","COMPLETE" if have("results-table") else "MISSING",
  "§2 — all numbers recomputed locally, stated in caption"),
 ("R2.a","emphasise novelty of the integration","WRITING","manuscript only"),
 ("R2.b","ablate GPT-4o-mini vs traditional / small LM",
  st(trained, have("target-acc-spacy")),
  "§4 parse-level evidence is complete; the five retrained arms are missing"),
 ("R2.c","report FLOPs, GPU memory, inference latency",
  ("COMPLETE" if (cx and all(list(cx.get("results",{}).values())[0].get(k) is not None
                             for k in ("flops","peak_memory_mb","latency_bs1")))
   else "PARTIAL" if have("complexity-model") else "MISSING"),
  "§8 — GFLOPs, peak memory and synchronised latency now measured"),
 ("R2.d","include failure cases","COMPLETE" if have("failure-cases") else "MISSING",
  "§6 — 6 rendered cases + full cause distribution"),
 ("R2.e","analyse how parse errors propagate",
  ("COMPLETE" if have("error-propagation") else
   "PARTIAL" if have("parse-quality-split") else "MISSING"),
  "§5 correlational + §5b causal corruption sweep with a flat control"),
 ("R2.f","shorten standard-architecture descriptions","WRITING","manuscript only"),
 ("R2.g","dedicated copy-paste ablation","COMPLETE" if have("train-no-copypaste") else "MISSING",
  "§2b — arm ran; contribution measured NEGATIVE, and seed-confounded"),
 ("R2.h","cite image-captioning works","WRITING","manuscript only"),
 ("R3.1","could conventional parsers / small LMs match?",
  st(trained, have("parse-field-comparison")),
  "§4 parse-level + §2b spaCy arm retrained; llama/smalllm arms still missing"),
 ("R3.2","broader comparison with recent baselines","WRITING","needs external numbers"),
 ("R3.3","justify hyper-parameters (layers, neighbours, depth)",
  "COMPLETE" if have("train-attention-sweep") else "MISSING",
  "train-attention-sweep never run (GPU)"),
 ("R3.4","the code repository is empty","COMPLETE",
  "repo is now substantial and documented — publish it; needs no compute"),
 ("R3.5","cite remote-sensing grounding works","WRITING","manuscript only"),
 ("R4.1","position vs LA-3D and TDBU-3DVG","WRITING","manuscript only"),
 ("R4.2","analyse by sentence length and relation type",
  "COMPLETE" if have("linguistic-complexity") else "MISSING",
  "§3 — done, but the widening trend is NOT significant; soften the claim"),
 ("R4.3","manual parse-quality assessment on a subset",
  "PARTIAL" if have("error-taxonomy") else "MISSING",
  "§6.2 — gpt only, n=30; needs human labelling, not compute"),
 ("R4.4","controlled comparison, parser varied, fusion fixed",
  st(trained), "§2b spacy+none retrained, fusion fixed; §5c adds the test-time swap"),
 ("R4.5","is parsing offline? report its cost",
  "COMPLETE" if have("parsing-offline-check") else "MISSING",
  "§8.1 — verdict OFFLINE, no parser in the forward path"),
 ("R4.6","sign of Eq. 7","COMPLETE" if have("verify-eq7") else "MISSING",
  "§7 — sign confirmed correct"),
 ("R4.7","separate the copy-paste contribution","COMPLETE" if have("train-no-copypaste") else "MISSING",
  "§2b — measured, but the sign is unfavourable; re-run seed 42 before quoting"),
 ("R4.8","results over multiple random seeds","COMPLETE" if have("train-seeds") else "MISSING",
  "§2b — 46.06 ± 0.05 pp over 2 seeds; a third seed would strengthen it"),
 ("R4.9","were Table 1 baselines reproduced identically?",
  "COMPLETE" if have("results-table") else "MISSING",
  "§2 — recomputed locally from predictions.p"),
]

ICON = {"COMPLETE": "DONE", "PARTIAL": "PARTIAL", "MISSING": "BLOCKED", "WRITING": "TEXT"}
rows = [[f"**{r}**", t, f"**{ICON[s]}**", e] for r, t, s, e in REVIEW]
mdtable(["point", "concern", "status", "evidence / blocker"], rows,
        name="10_reviewer_coverage",
        caption="Reviewer-by-reviewer coverage of the revision as it stands.")

from collections import Counter
cnt = Counter(s for *_, s, _ in REVIEW)
NUMBERS["reviewer_coverage"] = {r: s for r, _, s, _ in REVIEW}
NUMBERS["reviewer_summary"] = dict(cnt)
_blocked = [r for r, _, st_, _ in REVIEW if st_ == "MISSING"]
_partial = [r for r, _, st_, _ in REVIEW if st_ == "PARTIAL"]
both(f"""
### Tally

**{cnt['COMPLETE']} complete · {cnt['PARTIAL']} partial · {cnt['MISSING']} blocked · {cnt['WRITING']} manuscript-only**

Phase-A training has now run, and the shape of the revision has changed completely. The three
objections most likely to decide the resubmission — **R4.4** (the Table 6 defect), **R4.7**
(copy-paste) and **R4.8** (seed variance) — all now have measurements behind them, and **R2.c**
(FLOPs/memory/latency) and **R2.e** (causal error propagation) are closed too.

{"**Still blocked: " + ", ".join(_blocked) + "**." if _blocked else "**Nothing is fully blocked.**"}
{"Partial: " + ", ".join(_partial) + "." if _partial else ""}

What remains is no longer mostly a compute problem. The open items are:

- **{len(_blocked) + len(_partial)} experimental gap(s)** — chiefly the attention-layer sweep (R3.3)
  and the manual parse labelling (R4.3), the latter needing human effort rather than a GPU.
- **{cnt['WRITING']} manuscript-only points** — related work, novelty positioning and citations,
  which no experiment can address.
- **Three follow-up runs that would harden what already exists**: the crashed seed-42
  reference arm, a third seed, and the `llama`/`smalllm` training arms so §5c's train-vs-test
  comparison covers every parser.
""")

In [ ]:
cnt2 = Counter(s for *_, s, _ in REVIEW)
order = ["COMPLETE", "PARTIAL", "MISSING", "WRITING"]
cols = {"COMPLETE": "#009E73", "PARTIAL": "#E69F00", "MISSING": "#D55E00", "WRITING": "#BBBBBB"}
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.0),
                             gridspec_kw={"width_ratios": [1, 1.7]})
a1.bar([o.title() for o in order], [cnt2.get(o, 0) for o in order],
       color=[cols[o] for o in order], edgecolor="white")
for i, o in enumerate(order):
    a1.text(i, cnt2.get(o, 0) + 0.15, str(cnt2.get(o, 0)), ha="center", fontweight="bold")
a1.set_ylabel("reviewer points"); a1.set_title("Revision status")
a1.grid(axis="x", visible=False)

ys = np.arange(len(REVIEW))
a2.barh(ys, [1] * len(REVIEW), color=[cols[s] for *_, s, _ in REVIEW], edgecolor="white")
a2.set_yticks(ys)
a2.set_yticklabels([f"{r}  {t[:44]}{'…' if len(t) > 44 else ''}" for r, t, _, _ in REVIEW],
                   fontsize=7.6)
a2.invert_yaxis(); a2.set_xticks([]); a2.set_xlim(0, 1)
a2.set_title("Every reviewer point")
for sp in ("bottom", "left"): a2.spines[sp].set_visible(False)
a2.grid(visible=False)
handles = [plt.Rectangle((0, 0), 1, 1, color=cols[o]) for o in order]
a2.legend(handles, [o.title() for o in order], loc="lower right", ncol=4, fontsize=8)
fig.tight_layout()
fig_save(fig, "10_reviewer_coverage",
         "Coverage of all four reviewers' numbered concerns. Green is CPU analysis; red is "
         "GPU training.")

---
## 11 · Conclusions, and what is safe to claim

The distinction that matters when drafting the response: which statements the evidence
**supports**, which are **directional only**, and which would be **over-claiming** if a
reviewer checked.

In [ ]:
by = {"safe": [], "directional": [], "unsupported": []}
for s, t, e in CLAIMS:
    by.setdefault(s, []).append((t, e))

LABEL = {"safe": ("Safe to state as a finding", "#009E73"),
         "directional": ("Directional only — hedge the wording", "#E69F00"),
         "unsupported": ("Do NOT claim — the evidence does not support it", "#D55E00")}

for kind in ("safe", "directional", "unsupported"):
    items = by.get(kind, [])
    if not items: continue
    title, _ = LABEL[kind]
    h(f"{title}  ({len(items)})", 4)
    lines = []
    for t, e in items:
        lines.append(f"- {t}  \n  *evidence:* `{e}`")
    txt = "\n".join(lines)
    note(txt); report(f"**{title}**\n\n{txt}")

NUMBERS["claims"] = {k: [{"claim": t, "evidence": e} for t, e in v] for k, v in by.items()}

_arms = NUMBERS.get("ablation_arms", {})
_sd = NUMBERS.get("seed_spread", {}).get("sd_25")
both(f"""
### What to lead the response letter with

1. **The controlled parser ablation now exists.** With the fusion architecture held fixed and
   only the parse cache swapped, removing the parser costs
   {abs(pp(_arms.get('ABL-PARSER-NONE',{}).get('acc_0.25',0) - NUMBERS.get('seed_spread',{}).get('mean_25',0))):.2f} pp
   and substituting spaCy costs
   {abs(pp(_arms.get('ABL-PARSER-SPACY',{}).get('acc_0.25',0) - NUMBERS.get('seed_spread',{}).get('mean_25',0))):.2f} pp.
   This is precisely the design R4.4 said Table 6 lacked.
2. **Seed variance is measured: ±{pp(_sd):.2f} pp** over the main configuration. R4.8's
   objection — sub-one-point differences reported without run-to-run spread — is answerable,
   and it also gives every other ablation a noise floor to be judged against.
3. **Parse errors propagate causally, and the model is robust to them.** Corrupting half of
   all parses costs under a point globally, with a significant paired effect on the corrupted
   annotations and a flat untouched control. That is a stronger and more interesting answer to
   R2 than either "errors don't matter" or "the parser is critical".
4. **Eq. 7 is correct, and we measured it.** ρ(distance, attention) = −0.838 on the active
   head. R1.3 and R4.6 rest on a misreading of the sign.
5. **Cost is fully characterised**: {(cx or {}).get('results',{}).get('end2end',{}).get('flops',{}).get('total_gflops','--')} GFLOPs,
   peak memory and synchronised latency, closing R2.c.

### Still not answerable by analysis

**R3.3** (attention-layer sweep) needs its training runs, and **R4.3**'s manual parse
assessment needs human labelling of the three unlabelled sheets — no hardware involved.
""")

In [ ]:
# ======================================================================================
# Write everything to comparison/
# ======================================================================================
stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

# ---- numbers.json ----
def jsonable(o):
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)): return o.tolist()
    return o

NUMBERS["_meta"] = dict(generated=stamp, checkpoint=CHECKPOINT, split=SPLIT,
                        predictions_dated="2024-12-19 (original training run)",
                        artifacts_present=sum(1 for v in AVAIL.values() if v["ok"]),
                        artifacts_total=len(AVAIL))
(OUT / "numbers.json").write_text(json.dumps(jsonable(NUMBERS), indent=1))

# ---- SKIPPED.md ----
lines = [f"# Skipped analyses\n",
         f"_Generated {stamp}._\n",
         f"**{len(SKIPPED)} analyses were skipped** because their artifacts do not exist. Each "
         f"will populate automatically once the artifact appears — re-run this notebook.\n"]
if SKIPPED:
    lines += ["| experiment | missing artifact | would answer | blocked by |",
              "|---|---|---|---|"]
    for s in SKIPPED:
        lines.append(f"| `{s['key']}` | `{s['artifact']}` | {s['answers']} | {s['blocked_by']} |")
else:
    lines.append("Nothing was skipped — every artifact was present.")
lines += ["\n## Never-run experiments, by root cause\n"]

def coarse(blocker):
    """Collapse the specific blocker text into the handful of real root causes."""
    b = blocker.lower()
    if "small-lm parse cache" in b and "gpu" in b: return "Missing small-LM cache AND GPU"
    if "small-lm parse cache" in b:                return "Missing small-LM parse cache"
    if "needs corruption-sweep" in b or "needs train-seeds" in b:
        return "Depends on a GPU experiment that has not run"
    if "gpu" in b:                                 return "GPU unusable on this machine"
    if "never run" in b:                           return "Never invoked (no blocker)"
    return blocker

groups = {}
for key, phase, pat, answers, rev, blocker in REGISTRY:
    if not AVAIL[key]["ok"]:
        groups.setdefault(coarse(blocker), []).append((key, rev, answers))
for blocker, items in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    lines.append(f"\n### {blocker}  ({len(items)})\n")
    lines += ["| experiment | reviewer | what it would answer |", "|---|---|---|"]
    for k, rev, ans in items:
        lines.append(f"| `{k}` | {rev} | {ans} |")
(OUT / "SKIPPED.md").write_text("\n".join(lines) + "\n")

# ---- paper_tables.tex ----
tex = [f"% Generated {stamp} from comparison/numbers.json -- 3D-VG revision",
       "% Requires \\usepackage{booktabs}", ""]
if rt is not None:
    res, models = rt["results"], rt["models"]
    tex += [r"\begin{table}[t]", r"  \centering",
            r"  \caption{Grounding accuracy on ScanRefer " + f"({rt['split']}, "
            f"n={rt['n']}). All numbers recomputed with our evaluation code rather than "
            r"quoted from the original papers.}",
            r"  \label{tab:main}",
            r"  \begin{tabular}{lcccccc}", r"    \toprule",
            r"    & \multicolumn{2}{c}{Overall} & \multicolumn{2}{c}{Unique} & "
            r"\multicolumn{2}{c}{Multiple} \\",
            r"    \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}",
            r"    Method & @0.25 & @0.5 & @0.25 & @0.5 & @0.25 & @0.5 \\", r"    \midrule"]
    for m in models:
        v = [f"{pp(res[m][s][f'acc_{t}']):.2f}" for s in ("overall","unique","multiple")
             for t in ("0.25","0.5")]
        nm = r"\textbf{Ours}" if m == "ours" else m.replace("_", r"\_")
        cells = " & ".join((r"\textbf{" + x + "}") if m == "ours" else x for x in v)
        tex.append(f"    {nm} & {cells} " + r"\\")
    tex += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}", ""]
if lc is not None:
    for mk in [k for k in lc["results"] if k != "relation_types"][:2]:
        blk = lc["results"][mk]; models = list(blk["bins"][0]["models"])
        tex += [r"\begin{table}[t]", r"  \centering",
                r"  \caption{Accuracy@0.25 by " + blk["label"].replace("_", r"\_") +
                r". Per-bin gaps marked $^{*}$ are significant (McNemar, $p<0.05$); the "
                r"widening trend itself is not significant.}",
                r"  \label{tab:complexity-" + mk + "}",
                r"  \begin{tabular}{l" + "c" * (len(models) + 2) + "}", r"    \toprule",
                "    Bin & $n$ & " + " & ".join(
                    r"\textbf{Ours}" if m == "ours" else m for m in models) + r" & Gain \\",
                r"    \midrule"]
        det = list(blk.get("findings", {}).get("gaps", {}).values())
        det = det[0].get("per_bin_detail", []) if det else []
        for i, b in enumerate(blk["bins"]):
            vals = " & ".join(f"{pp(b['models'][m]['acc_0.25']):.2f}" for m in models)
            gap = pp(b["models"][models[0]]["acc_0.25"] - b["models"][models[1]]["acc_0.25"]) \
                  if len(models) > 1 else None
            mark = "$^{*}$" if i < len(det) and det[i]["p_value"] < 0.05 else ""
            tex.append(f"    {b['bin']} & {b['n']} & {vals} & "
                       + (f"{gap:+.2f}{mark}" if gap is not None else "--") + r" \\")
        tex += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}", ""]
if ta:
    real = {k: v for k, v in ta.items() if k != "none"}
    if real:
        tex += [r"\begin{table}[t]", r"  \centering",
                r"  \caption{Target-field extraction accuracy against ScanRefer's "
                r"\texttt{object\_name} (train split). Fuzzy matching absorbs synonyms.}",
                r"  \label{tab:parser-target}",
                r"  \begin{tabular}{lccc}", r"    \toprule",
                r"    Parser & Exact & Substring & Fuzzy \\", r"    \midrule"]
        for k in sorted(real, key=lambda k: -real[k]["splits"]["train"]["fuzzy_pct"]):
            s = real[k]["splits"]["train"]
            tex.append(f"    {k.replace('_', chr(92)+'_')} & {s['exact_pct']:.2f} & "
                       f"{s['substring_pct']:.2f} & {s['fuzzy_pct']:.2f} " + r"\\")
        tex += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}", ""]
# ---- ablation arms (phase A) ----
if "RUNS" in dir() and RUNS and ref_mean is not None:
    _order = [k for k, *_ in ARMS if k in RUNS]
    tex += [r"\begin{table}[t]", r"  \centering",
            r"  \caption{Ablation study. All arms train the fusion head on cached "
            r"detector proposals for 20 epochs from a shared warm start, so they are "
            r"comparable to each other but not to Table~\ref{tab:main}. $\Delta$ is against "
            r"the mean of the seed runs, whose spread is $\pm" + f"{pp(ref_sd):.2f}" +
            r"$~pp.}",
            r"  \label{tab:ablation}",
            r"  \begin{tabular}{llcccc}", r"    \toprule",
            r"    Arm & Parser & Acc@0.25 & Acc@0.5 & $\Delta$@0.25 & $|\Delta|/\sigma$ \\",
            r"    \midrule"]
    for k in _order:
        v = RUNS[k]
        d = pp(v["acc_0.25"] - ref_mean)
        nsd = abs(d) / pp(ref_sd) if pp(ref_sd) else float("inf")
        nm = k.replace("ABL-", "").replace("_", r"\_")
        tex.append(f"    {nm} & {v['parser']} & {pp(v['acc_0.25']):.2f} & "
                   f"{pp(v['acc_0.5']):.2f} & {d:+.2f} & "
                   + (f"{nsd:.1f}" if np.isfinite(nsd) else "--") + r" \\")
    tex += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}", ""]

# ---- efficiency ----
if cx is not None:
    _r = cx.get("results", {})
    if any(v.get("flops") for v in _r.values()):
        tex += [r"\begin{table}[t]", r"  \centering",
                r"  \caption{Computational cost, measured on this hardware. The cached "
                r"variant evaluates only the language and fusion stages, as the ablation "
                r"protocol does.}",
                r"  \label{tab:complexity}",
                r"  \begin{tabular}{lcccc}", r"    \toprule",
                r"    Variant & Params & GFLOPs & Peak mem. (MB) & Latency (ms) \\",
                r"    \midrule"]
        for vn, v in _r.items():
            f = v.get("flops") or {}
            pm = v.get("peak_memory_mb") or {}
            lt = v.get("latency_bs1") or {}
            tex.append(f"    {vn} & {v.get('params',{}).get('total',0):,} & "
                       f"{f.get('total_gflops','--')} & "
                       f"{pm.get('inference_bs1','--')} & "
                       + (f"{lt.get('mean_ms'):.1f} $\\pm$ {lt.get('std_ms'):.1f}"
                          if lt.get("mean_ms") is not None else "--") + r" \\")
        tex += [r"    \bottomrule", r"  \end{tabular}", r"\end{table}", ""]

(OUT / "paper_tables.tex").write_text("\n".join(tex) + "\n")

# ---- CONCLUSIONS.md ----
con = [f"# Conclusions\n", f"_Generated {stamp}._\n",
       f"Artifacts present: **{NUMBERS['_meta']['artifacts_present']}/"
       f"{NUMBERS['_meta']['artifacts_total']}**. "
       f"All accuracy numbers derive from `predictions.p` dated "
       f"**{NUMBERS['_meta']['predictions_dated']}**.\n"]
for kind in ("safe", "directional", "unsupported"):
    items = by.get(kind, [])
    if not items: continue
    con.append(f"\n## {LABEL[kind][0]}\n")
    for t, e in items:
        con.append(f"- {t}\n  - evidence: `{e}`")
if "reviewer_summary" in NUMBERS:
    s = NUMBERS["reviewer_summary"]
    con.append(f"\n## Reviewer tally\n\n"
               f"- complete: **{s.get('COMPLETE',0)}**\n- partial: **{s.get('PARTIAL',0)}**\n"
               f"- blocked: **{s.get('MISSING',0)}**\n- manuscript-only: **{s.get('WRITING',0)}**\n")
(OUT / "CONCLUSIONS.md").write_text("\n".join(con) + "\n")

# ---- REPORT.md ----
rep = [f"# 3D-VG — experiment results report\n", f"_Generated {stamp}._\n",
       f"- checkpoint: `{CHECKPOINT}`\n- split: `{SPLIT}`\n"
       f"- artifacts present: **{NUMBERS['_meta']['artifacts_present']}/"
       f"{NUMBERS['_meta']['artifacts_total']}**\n"
       f"- predictions dated: **{NUMBERS['_meta']['predictions_dated']}**\n",
       "> Every accuracy number below comes from the original December 2024 training run. "
       "Retraining invalidates them; re-run this notebook afterwards.\n", "---\n"]
rep += ["\n\n---\n\n".join(REPORT)]
if FIGURES:
    rep += ["\n\n---\n\n## Figures\n"]
    for name, path, cap in FIGURES:
        rep.append(f"### {name}\n\n![{cap}]({Path(path).relative_to('comparison')})\n\n{cap}\n")
if SKIPPED:
    rep += [f"\n---\n\n## Skipped ({len(SKIPPED)})\n",
            "See `SKIPPED.md` for the full breakdown by cause.\n",
            "| experiment | would answer | blocked by |", "|---|---|---|"]
    for s in SKIPPED:
        rep.append(f"| `{s['key']}` | {s['answers']} | {s['blocked_by']} |")
(OUT / "REPORT.md").write_text("\n".join(rep) + "\n")

# ---- summary ----
display(Markdown(f"""
### Written to `comparison/`

| file | what |
|---|---|
| `REPORT.md` | the full narrative report ({len(REPORT)} blocks) |
| `CONCLUSIONS.md` | claims split into safe / directional / unsupported |
| `SKIPPED.md` | {len(SKIPPED)} skipped analyses, grouped by cause |
| `numbers.json` | every number, machine-readable |
| `paper_tables.tex` | {sum(1 for l in tex if l.startswith(chr(92)+'begin{table}'))} LaTeX tables, booktabs |
| `availability.csv` | {len(AVAIL)} experiments × availability |
| `figures/` | {len(FIGURES)} PNG at 200 dpi |
| `tables/` | {len(TABLES)} Markdown tables |
"""))
for name, path, _ in FIGURES: print(f"  fig    {path}")
print()
for f in ["REPORT.md", "CONCLUSIONS.md", "SKIPPED.md", "numbers.json",
          "paper_tables.tex", "availability.csv"]:
    p = OUT / f
    print(f"  {'ok ' if p.is_file() else 'MISS'}   comparison/{f}"
          f"   {p.stat().st_size/1024:.1f} KB" if p.is_file() else f"  MISSING comparison/{f}")

---
## Notes

**Re-running.** Everything is derived, so *Run All* is always safe and takes seconds. When a
blocked experiment finally produces its artifact, re-run this notebook: the availability
table in §1 picks it up, the matching section stops skipping, and `comparison/` is rewritten.

**What this notebook deliberately does not do.** It never recomputes accuracy from
`predictions.p` itself — the analysis scripts under `experiments/analysis/` own that, and
duplicating their statistics here would create two sources of truth that could silently
disagree. This notebook aggregates, cross-compares, corrects for multiple comparisons, and
draws conclusions.

**The staleness trap worth remembering.** `outputs/reports/RUN_REPORT.md` claims
*"0 succeeded, 0 skipped"* while `outputs/analysis/` is full of results — it was rendered
from an empty `results` dict. The `plan` field of `outputs/run_analysis_summary.json` is
accurate; its `results` field is not. This notebook reads the artifacts directly and ignores
both.

**Related notebooks**

| notebook | for |
|---|---|
| `run_analysis_colab.ipynb` | run all 36 experiments in dependency order |
| `run_one_experiment.ipynb` | run exactly one, with a confirmation gate |
| `run_parser_analysis.ipynb` | the two parser-field analyses, with inline figures |
| `run_smalllm_parsing.ipynb` | build a small-LM parse cache (variant E) |